# Archive - Final Pipeline Backup Before Presentation Edit

Archived copy preserved for project history. Outputs are cleared and local/HPC paths are sanitized for the public repository.

**Portfolio note:** This notebook is part of a cleaned public portfolio version. Large datasets, trained model weights, HPC runtime artifacts, and private machine paths are not included.

**Public repository sections:**

- Archive Note


## Section 0 — Experiment Goal And Save Path

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".")
DATA_ROOT = PROJECT_ROOT / "Universal_Tomato_Dataset"
PREV_EXPERIMENT_ROOT = PROJECT_ROOT / "trackB_outputs" / "Track_B_Experiment_2"
EXPERIMENT_ROOT = PROJECT_ROOT / "trackB_outputs" / "Track_B_Experiment_3"

SAVE_DIRS = {
    "root": EXPERIMENT_ROOT,
    "configs": EXPERIMENT_ROOT / "configs",
    "audits": EXPERIMENT_ROOT / "audits",
    "leaf_gate": EXPERIMENT_ROOT / "leaf_gate",
    "disease_classifier": EXPERIMENT_ROOT / "disease_classifier",
    "calibration": EXPERIMENT_ROOT / "calibration",
    "thresholds": EXPERIMENT_ROOT / "thresholds",
    "reports": EXPERIMENT_ROOT / "reports",
    "figures": EXPERIMENT_ROOT / "figures",
    "predictions": EXPERIMENT_ROOT / "predictions",
    "deployment": EXPERIMENT_ROOT / "deployment",
    "comparisons": EXPERIMENT_ROOT / "comparisons",
}

for path in SAVE_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")

print("Project root        :", PROJECT_ROOT)
print("Dataset root        :", DATA_ROOT)
print("Prev experiment root:", PREV_EXPERIMENT_ROOT)
print("Experiment root     :", EXPERIMENT_ROOT)
print("Save directories    :")
for name, path in SAVE_DIRS.items():
    print(f"  - {name:18s} -> {path}")


## Section 1 — Environment, GPU, And Local Backbone Cache

In [ ]:
# Uncomment only if this kernel is missing packages.
# %pip install -U pip ipywidgets timm albumentations scikit-learn pandas numpy matplotlib seaborn pillow safetensors
# %pip install --index-url https://download.pytorch.org/whl/cu130 torch torchvision

import os
import torch
import torchvision
import timm

# Silence optional HF progress noise. We will use local cached weights anyway.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this notebook kernel. "
        "Select the GPU-enabled .venv312 kernel before continuing."
    )

DEVICE = torch.device("cuda")
gpu_props = torch.cuda.get_device_properties(0)
total_mem_gb = gpu_props.total_memory / (1024 ** 3)

LOCAL_BACKBONE_CACHE = {
    "tf_efficientnet_b3": Path(
        "<HPC_HOME_REMOVED>/.cache/huggingface/hub/"
        "models--timm--tf_efficientnet_b3.ns_jft_in1k/"
        "snapshots/c535b7f55accd867106cecfa1c96ac984b8348fe/model.safetensors"
    ),
    "tf_efficientnet_b0": Path(
        "<HPC_HOME_REMOVED>/.cache/huggingface/hub/"
        "models--timm--tf_efficientnet_b0.ns_jft_in1k/"
        "snapshots/625afb926af7ee851c12bdfef4520d7b2ba679d5/model.safetensors"
    ),
}

missing_local_backbones = [name for name, path in LOCAL_BACKBONE_CACHE.items() if not path.exists()]
if missing_local_backbones:
    raise FileNotFoundError(
        "Missing local timm backbone cache for: "
        f"{missing_local_backbones}. "
        "Experiment 3 is configured to use local cached weights to avoid HF warnings."
    )

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("timm version       :", timm.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("CUDA device        :", torch.cuda.get_device_name(0))
print("GPU total memory   :", f"{total_mem_gb:.2f} GB")
print("Device             :", DEVICE)

print("\nLocal backbone cache:")
for name, path in LOCAL_BACKBONE_CACHE.items():
    print(f"  - {name:20s} -> {path}")


## Section 2 — Imports, Seeds, And Warning-Free Runtime

In [ ]:
import copy
import json
import math
import random
import time
import warnings
from collections import Counter
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from typing import Dict, List, Optional, Tuple

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import torch.optim as optim

from IPython.display import display
from PIL import Image, ImageFile
from safetensors.torch import load_file as safe_load_file

from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
)
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)

AMP_DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"
SCALER = GradScaler(device=AMP_DEVICE_TYPE, enabled=torch.cuda.is_available())

def autocast_ctx():
    if AMP_DEVICE_TYPE == "cuda":
        return autocast(device_type="cuda", enabled=SCALER.is_enabled())
    return nullcontext()

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def configure_torch_runtime() -> None:
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

def strip_common_prefixes(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith("module."):
            key = key[len("module."):]
        if key.startswith("model."):
            key = key[len("model."):]
        cleaned[key] = value
    return cleaned

def load_checkpoint_state_dict(checkpoint_path: str) -> Dict[str, torch.Tensor]:
    checkpoint_path = str(checkpoint_path)
    suffix = Path(checkpoint_path).suffix.lower()

    if suffix == ".safetensors":
        state_dict = safe_load_file(checkpoint_path)
        return strip_common_prefixes(state_dict)

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model_state_dict", "model", "ema_state_dict"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return strip_common_prefixes(checkpoint[key])

    if isinstance(checkpoint, dict):
        return strip_common_prefixes(checkpoint)

    raise RuntimeError(f"Unsupported checkpoint format: {checkpoint_path}")

def load_local_pretrained_weights(
    model: nn.Module,
    checkpoint_path: str,
    verbose_name: str,
) -> Dict[str, object]:
    state_dict = load_checkpoint_state_dict(checkpoint_path)
    model_state = model.state_dict()

    compatible_state_dict = {}
    skipped_keys = []

    for key, value in state_dict.items():
        if key in model_state and tuple(model_state[key].shape) == tuple(value.shape):
            compatible_state_dict[key] = value
        else:
            skipped_keys.append(key)

    incompatible = model.load_state_dict(compatible_state_dict, strict=False)

    report = {
        "loaded_keys": len(compatible_state_dict),
        "skipped_keys": len(skipped_keys),
        "missing_after_load": len(incompatible.missing_keys),
        "unexpected_after_load": len(incompatible.unexpected_keys),
    }

    print(
        f"{verbose_name} local pretrained load | "
        f"loaded={report['loaded_keys']} | "
        f"skipped={report['skipped_keys']} | "
        f"missing_after_load={report['missing_after_load']} | "
        f"unexpected_after_load={report['unexpected_after_load']}"
    )
    return report

seed_everything(42)
configure_torch_runtime()

print("Seeds initialized.")
print("AMP device type :", AMP_DEVICE_TYPE)
print("AMP enabled     :", SCALER.is_enabled())
print("CPU workers     :", os.cpu_count())


## Section 3 — Master Config V3

In [ ]:
@dataclass
class CFG:
    # Paths
    project_root: str = str(PROJECT_ROOT)
    data_root: str = str(DATA_ROOT)
    prev_experiment_root: str = str(PREV_EXPERIMENT_ROOT)
    experiment_root: str = str(EXPERIMENT_ROOT)
    train_dir: str = "train"
    val_dir: str = "val"
    test_dir: str = "test"

    # Local pretrained checkpoints
    disease_pretrained_checkpoint: str = str(LOCAL_BACKBONE_CACHE["tf_efficientnet_b3"])
    leaf_gate_pretrained_checkpoint: str = str(LOCAL_BACKBONE_CACHE["tf_efficientnet_b0"])

    # Models
    disease_model_name: str = "tf_efficientnet_b3"
    leaf_gate_model_name: str = "tf_efficientnet_b0"
    num_classes: int = 16
    drop_path_rate: float = 0.15

    # Image sizes
    disease_img_size: int = 320
    leaf_gate_img_size: int = 224

    # Disease training
    disease_epochs: int = 40
    mixup_off_last_n_epochs: int = 6
    disease_batch_size: int = 8
    grad_accum_steps: int = 2
    disease_lr: float = 4e-5
    finetune_lr: float = 1.2e-5
    weight_decay: float = 5e-4
    min_lr: float = 1e-6
    warmup_epochs: int = 3
    label_smoothing: float = 0.05
    grad_clip: float = 1.0
    disease_patience: int = 8

    # Mix regularization
    use_mixup: bool = True
    mixup_alpha: float = 0.2
    use_cutmix: bool = True
    cutmix_alpha: float = 0.8
    mix_prob: float = 0.6

    # Sampling / class weighting
    use_weighted_sampler: bool = False
    class_weight_power: float = 0.5

    # EMA
    use_ema: bool = True
    ema_decay: float = 0.9998

    # Leaf gate
    leaf_gate_epochs: int = 8
    leaf_gate_batch_size: int = 16
    leaf_gate_lr: float = 1e-4
    leaf_gate_patience: int = 4
    synth_negative_ratio: float = 1.0

    # Calibration / inference
    use_temperature_scaling: bool = True
    use_tta: bool = True
    tta_rounds: int = 7
    top_k: int = 3
    min_leaf_confidence: float = 0.50
    min_disease_confidence: float = 0.75
    uncertainty_margin: float = 0.05

    # System
    num_workers: int = min(4, os.cpu_count() or 2)
    pin_memory: bool = True
    persistent_workers: bool = True
    device: str = str(DEVICE)
    seed: int = 42

cfg = CFG()

if cfg.mixup_off_last_n_epochs >= cfg.disease_epochs:
    raise ValueError("mixup_off_last_n_epochs must be smaller than disease_epochs")

config_path = SAVE_DIRS["configs"] / "initial_config_v3.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2)

print(cfg)
print("Config saved to:", config_path)
print("\nExperiment 3 baseline:")
print("  disease_model_name       :", cfg.disease_model_name)
print("  disease_img_size         :", cfg.disease_img_size)
print("  disease_lr               :", cfg.disease_lr)
print("  mixup_off_last_n_epochs  :", cfg.mixup_off_last_n_epochs)
print("  use_weighted_sampler     :", cfg.use_weighted_sampler)
print("  use_ema                  :", cfg.use_ema)


## Section 4 — Dataset Audit And Split-Shift Diagnostics

In [ ]:
import cv2

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMG_EXTS

def collect_dataset_records(cfg: CFG) -> pd.DataFrame:
    records = []

    for split in [cfg.train_dir, cfg.val_dir, cfg.test_dir]:
        split_root = Path(cfg.data_root) / split
        if not split_root.exists():
            raise FileNotFoundError(f"Split path not found: {split_root}")

        class_dirs = sorted([p for p in split_root.iterdir() if p.is_dir()])
        if not class_dirs:
            raise RuntimeError(f"No class folders found in: {split_root}")

        for class_dir in class_dirs:
            image_paths = sorted([p for p in class_dir.iterdir() if is_image_file(p)])
            for path in image_paths:
                records.append({
                    "split": split,
                    "class_name": class_dir.name,
                    "path": str(path),
                    "file_name": path.name,
                    "suffix": path.suffix.lower(),
                })

    records_df = pd.DataFrame(records)
    if records_df.empty:
        raise RuntimeError("No images found in dataset.")
    return records_df

def sample_image_size_stats(records_df: pd.DataFrame, sample_per_split: int = 1200) -> pd.DataFrame:
    sampled_chunks = []

    for split, split_df in records_df.groupby("split"):
        n = min(sample_per_split, len(split_df))
        sampled = split_df.sample(n=n, random_state=42).copy()

        widths, heights, ratios, bad_paths = [], [], [], []

        for path_str in sampled["path"].tolist():
            try:
                with Image.open(path_str) as img:
                    w, h = img.size
                widths.append(w)
                heights.append(h)
                ratios.append(w / max(h, 1))
                bad_paths.append("")
            except Exception as exc:
                widths.append(np.nan)
                heights.append(np.nan)
                ratios.append(np.nan)
                bad_paths.append(str(exc))

        sampled["width"] = widths
        sampled["height"] = heights
        sampled["aspect_ratio"] = ratios
        sampled["read_error"] = bad_paths
        sampled_chunks.append(sampled)

    return pd.concat(sampled_chunks, ignore_index=True)

def compute_split_shift_diagnostics(records_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    split_order = [cfg.train_dir, cfg.val_dir, cfg.test_dir]

    global_split_counts = records_df["split"].value_counts().reindex(split_order, fill_value=0)
    global_split_share = (global_split_counts / global_split_counts.sum()).to_dict()

    class_split_counts = (
        records_df.groupby(["class_name", "split"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=split_order, fill_value=0)
    )

    class_split_share = class_split_counts.div(class_split_counts.sum(axis=1), axis=0)

    ratio_rows = []
    drift_rows = []

    for class_name in class_split_counts.index:
        count_row = class_split_counts.loc[class_name]
        share_row = class_split_share.loc[class_name]

        ratio_rows.append({
            "class_name": class_name,
            "train_count": int(count_row[cfg.train_dir]),
            "val_count": int(count_row[cfg.val_dir]),
            "test_count": int(count_row[cfg.test_dir]),
            "train_share_within_class": float(share_row[cfg.train_dir]),
            "val_share_within_class": float(share_row[cfg.val_dir]),
            "test_share_within_class": float(share_row[cfg.test_dir]),
        })

        deltas = {
            split: float(abs(share_row[split] - global_split_share[split]))
            for split in split_order
        }

        dominant_shift_split = max(deltas, key=deltas.get)

        drift_rows.append({
            "class_name": class_name,
            "global_train_share": float(global_split_share[cfg.train_dir]),
            "global_val_share": float(global_split_share[cfg.val_dir]),
            "global_test_share": float(global_split_share[cfg.test_dir]),
            "class_train_share": float(share_row[cfg.train_dir]),
            "class_val_share": float(share_row[cfg.val_dir]),
            "class_test_share": float(share_row[cfg.test_dir]),
            "max_abs_share_delta": float(max(deltas.values())),
            "dominant_shift_split": dominant_shift_split,
        })

    split_ratio_df = pd.DataFrame(ratio_rows).sort_values("class_name").reset_index(drop=True)
    split_shift_df = pd.DataFrame(drift_rows).sort_values(
        "max_abs_share_delta", ascending=False
    ).reset_index(drop=True)

    return {
        "split_ratio_df": split_ratio_df,
        "split_shift_df": split_shift_df,
    }

def run_dataset_audit(cfg: CFG) -> Dict[str, pd.DataFrame]:
    audits_dir = SAVE_DIRS["audits"]
    figures_dir = SAVE_DIRS["figures"]

    records_df = collect_dataset_records(cfg)
    records_df.to_csv(audits_dir / "dataset_records.csv", index=False)

    split_summary_df = (
        records_df.groupby("split")
        .agg(
            num_images=("path", "size"),
            num_classes=("class_name", "nunique"),
        )
        .reset_index()
        .sort_values("split")
    )
    split_summary_df.to_csv(audits_dir / "split_summary.csv", index=False)

    split_class_counts_df = (
        records_df.groupby(["split", "class_name"])
        .size()
        .reset_index(name="count")
        .sort_values(["split", "class_name"])
    )
    split_class_counts_df.to_csv(audits_dir / "split_class_counts.csv", index=False)

    train_counts_df = (
        split_class_counts_df[split_class_counts_df["split"] == cfg.train_dir]
        .copy()
        .rename(columns={"count": "train_count"})
        .sort_values("train_count", ascending=False)
    )

    max_count = train_counts_df["train_count"].max()
    min_count = train_counts_df["train_count"].min()

    imbalance_df = train_counts_df.copy()
    imbalance_df["max_to_class_ratio"] = max_count / imbalance_df["train_count"]
    imbalance_df["class_to_min_ratio"] = imbalance_df["train_count"] / max(min_count, 1)
    imbalance_df.to_csv(audits_dir / "train_imbalance_report.csv", index=False)

    filename_collision_df = (
        records_df.groupby("file_name")
        .agg(
            num_occurrences=("file_name", "size"),
            splits=("split", lambda x: ",".join(sorted(set(x)))),
            classes=("class_name", lambda x: ",".join(sorted(set(x)))),
        )
        .reset_index()
    )
    filename_collision_df = filename_collision_df[filename_collision_df["num_occurrences"] > 1]
    filename_collision_df.to_csv(audits_dir / "filename_collisions.csv", index=False)

    image_size_samples_df = sample_image_size_stats(records_df, sample_per_split=1200)
    image_size_samples_df.to_csv(audits_dir / "image_size_samples.csv", index=False)

    image_size_summary_df = (
        image_size_samples_df.groupby("split")
        .agg(
            sampled_images=("path", "size"),
            mean_width=("width", "mean"),
            mean_height=("height", "mean"),
            min_width=("width", "min"),
            min_height=("height", "min"),
            max_width=("width", "max"),
            max_height=("height", "max"),
            mean_aspect_ratio=("aspect_ratio", "mean"),
        )
        .reset_index()
    )
    image_size_summary_df.to_csv(audits_dir / "image_size_summary.csv", index=False)

    shift_artifacts = compute_split_shift_diagnostics(records_df)
    shift_artifacts["split_ratio_df"].to_csv(audits_dir / "split_ratio_by_class.csv", index=False)
    shift_artifacts["split_shift_df"].to_csv(audits_dir / "split_shift_summary.csv", index=False)

    plt.figure(figsize=(14, 6))
    sns.barplot(data=split_class_counts_df, x="class_name", y="count", hue="split")
    plt.title("Class Distribution Across Splits")
    plt.xlabel("Class")
    plt.ylabel("Image Count")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "class_distribution_all_splits.png", dpi=200)
    plt.show()

    plt.figure(figsize=(14, 6))
    sns.barplot(data=imbalance_df, x="class_name", y="train_count")
    plt.title("Training Split Class Imbalance")
    plt.xlabel("Class")
    plt.ylabel("Train Count")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "train_class_imbalance.png", dpi=200)
    plt.show()

    print("Dataset audit completed.")
    print("Total images:", len(records_df))

    print("\nSplit summary:")
    display(split_summary_df)

    print("\nMost imbalanced training classes:")
    display(imbalance_df.sort_values("train_count").head(8))

    print("\nLargest class-level split-share drift:")
    display(shift_artifacts["split_shift_df"].head(8))

    print("\nImage size summary:")
    display(image_size_summary_df)

    print("\nAudit files saved under:", audits_dir)

    return {
        "records": records_df,
        "split_summary": split_summary_df,
        "split_class_counts": split_class_counts_df,
        "imbalance": imbalance_df,
        "filename_collisions": filename_collision_df,
        "image_size_samples": image_size_samples_df,
        "image_size_summary": image_size_summary_df,
        "split_ratio_df": shift_artifacts["split_ratio_df"],
        "split_shift_df": shift_artifacts["split_shift_df"],
    }

audit_artifacts = run_dataset_audit(cfg)


## Section 5 — Disease Augmentation Policy V3

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def get_disease_train_tfms(img_size: int):
    return A.Compose([
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.78, 1.00),
            ratio=(0.88, 1.12),
            interpolation=cv2.INTER_LINEAR,
            p=1.0,
        ),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.96, 1.04),
            translate_percent=(-0.04, 0.04),
            rotate=(-10, 10),
            shear=(-3, 3),
            interpolation=cv2.INTER_LINEAR,
            p=0.35,
        ),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.16,
                contrast_limit=0.16,
                p=1.0,
            ),
            A.HueSaturationValue(
                hue_shift_limit=6,
                sat_shift_limit=12,
                val_shift_limit=10,
                p=1.0,
            ),
            A.ColorJitter(
                brightness=0.12,
                contrast=0.12,
                saturation=0.12,
                hue=0.04,
                p=1.0,
            ),
        ], p=0.45),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=(3, 5), p=1.0),
            A.ImageCompression(quality_range=(45, 90), p=1.0),
            A.GaussNoise(std_range=(0.01, 0.03), p=1.0),
        ], p=0.20),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.05, 0.14),
            hole_width_range=(0.05, 0.14),
            fill=0,
            p=0.10,
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_disease_eval_tfms(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_leaf_gate_positive_train_tfms(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.96, 1.04),
            translate_percent=(-0.03, 0.03),
            rotate=(-8, 8),
            shear=(-2, 2),
            interpolation=cv2.INTER_LINEAR,
            p=0.30,
        ),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.12,
                contrast_limit=0.12,
                p=1.0,
            ),
            A.HueSaturationValue(
                hue_shift_limit=5,
                sat_shift_limit=8,
                val_shift_limit=6,
                p=1.0,
            ),
        ], p=0.30),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_leaf_gate_negative_post_train_tfms():
    return A.Compose([
        A.OneOf([
            A.ImageCompression(quality_range=(25, 65), p=1.0),
            A.GaussNoise(std_range=(0.01, 0.05), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.30),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_leaf_gate_eval_tfms(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_leaf_gate_negative_post_eval_tfms():
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

print("Experiment 3 augmentations are ready.")


## Section 6 — Leaf Gate Setup V3

In [ ]:
LEAF_GATE_NEGATIVE_MODES = [
    "tiny_crop",
    "edge_patch",
    "occluded",
    "degraded",
]

def load_rgb_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        return np.array(img.convert("RGB"))

def _safe_resize(image: np.ndarray, img_size: int) -> np.ndarray:
    return cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

def synth_tiny_crop_negative(image: np.ndarray, img_size: int, rng: np.random.Generator) -> np.ndarray:
    h, w = image.shape[:2]
    crop_scale = float(rng.uniform(0.08, 0.20))
    crop_h = max(12, int(h * crop_scale))
    crop_w = max(12, int(w * crop_scale))

    y0 = int(rng.integers(0, max(1, h - crop_h + 1)))
    x0 = int(rng.integers(0, max(1, w - crop_w + 1)))

    crop = image[y0:y0 + crop_h, x0:x0 + crop_w]
    crop = _safe_resize(crop, img_size)
    crop = cv2.GaussianBlur(crop, (11, 11), 0)
    return crop

def synth_edge_patch_negative(image: np.ndarray, img_size: int, rng: np.random.Generator) -> np.ndarray:
    h, w = image.shape[:2]
    side = rng.choice(["top", "bottom", "left", "right"])
    band_ratio = float(rng.uniform(0.08, 0.18))

    if side in ["top", "bottom"]:
        band = max(10, int(h * band_ratio))
        patch = image[:band, :, :] if side == "top" else image[h - band:, :, :]
    else:
        band = max(10, int(w * band_ratio))
        patch = image[:, :band, :] if side == "left" else image[:, w - band:, :]

    patch = _safe_resize(patch, img_size)
    patch = cv2.GaussianBlur(patch, (9, 9), 0)
    return patch

def synth_occluded_negative(image: np.ndarray, img_size: int, rng: np.random.Generator) -> np.ndarray:
    out = _safe_resize(image, img_size).copy()

    num_boxes = int(rng.integers(6, 12))
    for _ in range(num_boxes):
        box_h = int(rng.integers(max(12, img_size // 10), max(16, img_size // 3)))
        box_w = int(rng.integers(max(12, img_size // 10), max(16, img_size // 3)))
        y0 = int(rng.integers(0, max(1, img_size - box_h + 1)))
        x0 = int(rng.integers(0, max(1, img_size - box_w + 1)))
        fill_value = int(rng.integers(0, 256))
        out[y0:y0 + box_h, x0:x0 + box_w] = fill_value

    out = cv2.GaussianBlur(out, (7, 7), 0)
    return out

def synth_degraded_negative(image: np.ndarray, img_size: int, rng: np.random.Generator) -> np.ndarray:
    out = _safe_resize(image, img_size)
    gray = cv2.cvtColor(out, cv2.COLOR_RGB2GRAY)
    out = np.stack([gray, gray, gray], axis=-1)

    small_side = int(rng.integers(max(24, img_size // 8), max(32, img_size // 4)))
    down = cv2.resize(out, (small_side, small_side), interpolation=cv2.INTER_AREA)
    out = cv2.resize(down, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    noise = rng.normal(0, 18, size=out.shape).astype(np.float32)
    out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    out = cv2.GaussianBlur(out, (9, 9), 0)
    return out

def make_synthetic_negative_image(
    image: np.ndarray,
    img_size: int,
    mode: str,
    seed: Optional[int] = None,
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    if mode == "tiny_crop":
        return synth_tiny_crop_negative(image, img_size, rng)
    if mode == "edge_patch":
        return synth_edge_patch_negative(image, img_size, rng)
    if mode == "occluded":
        return synth_occluded_negative(image, img_size, rng)
    if mode == "degraded":
        return synth_degraded_negative(image, img_size, rng)

    raise ValueError(f"Unknown synthetic negative mode: {mode}")

def build_leaf_gate_manifest(cfg: CFG, records_df: Optional[pd.DataFrame] = None) -> Dict[str, pd.DataFrame]:
    if records_df is None:
        records_df = collect_dataset_records(cfg)

    manifest_rows = []

    for split in [cfg.train_dir, cfg.val_dir, cfg.test_dir]:
        split_df = records_df[records_df["split"] == split].reset_index(drop=True)

        for idx, row in split_df.iterrows():
            manifest_rows.append({
                "split": split,
                "path": row["path"],
                "class_name": row["class_name"],
                "leaf_label": 1,
                "sample_type": "positive",
                "negative_mode": "",
                "deterministic_seed": idx,
            })

            neg_repeats = 1 if split != cfg.train_dir else max(1, int(round(cfg.synth_negative_ratio)))
            for neg_idx in range(neg_repeats):
                negative_mode = LEAF_GATE_NEGATIVE_MODES[(idx + neg_idx) % len(LEAF_GATE_NEGATIVE_MODES)]
                manifest_rows.append({
                    "split": split,
                    "path": row["path"],
                    "class_name": row["class_name"],
                    "leaf_label": 0,
                    "sample_type": "synthetic_negative",
                    "negative_mode": negative_mode,
                    "deterministic_seed": idx + neg_idx,
                })

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_path = SAVE_DIRS["leaf_gate"] / "leaf_gate_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)

    manifest_summary_df = (
        manifest_df.groupby(["split", "sample_type", "leaf_label"])
        .size()
        .reset_index(name="count")
        .sort_values(["split", "sample_type"])
    )
    manifest_summary_df.to_csv(SAVE_DIRS["leaf_gate"] / "leaf_gate_manifest_summary.csv", index=False)

    print("Leaf gate manifest saved to:", manifest_path)
    display(manifest_summary_df)

    return {
        "manifest": manifest_df,
        "train": manifest_df[manifest_df["split"] == cfg.train_dir].reset_index(drop=True),
        "val": manifest_df[manifest_df["split"] == cfg.val_dir].reset_index(drop=True),
        "test": manifest_df[manifest_df["split"] == cfg.test_dir].reset_index(drop=True),
        "summary": manifest_summary_df,
    }

def preview_synthetic_leaf_gate_examples(cfg: CFG, records_df: pd.DataFrame, n_per_mode: int = 2):
    train_records = records_df[records_df["split"] == cfg.train_dir].reset_index(drop=True)
    sample_rows = train_records.sample(
        n=min(len(LEAF_GATE_NEGATIVE_MODES) * n_per_mode, len(train_records)),
        random_state=cfg.seed,
    ).reset_index(drop=True)

    num_rows = len(sample_rows)
    num_cols = 1 + len(LEAF_GATE_NEGATIVE_MODES)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 3.2 * num_rows))
    if num_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, sample) in enumerate(sample_rows.iterrows()):
        image = load_rgb_image(sample["path"])

        axes[row_idx, 0].imshow(_safe_resize(image, cfg.leaf_gate_img_size))
        axes[row_idx, 0].set_title("positive")
        axes[row_idx, 0].axis("off")

        for col_idx, mode in enumerate(LEAF_GATE_NEGATIVE_MODES, start=1):
            neg_img = make_synthetic_negative_image(
                image=image,
                img_size=cfg.leaf_gate_img_size,
                mode=mode,
                seed=row_idx + col_idx,
            )
            axes[row_idx, col_idx].imshow(neg_img)
            axes[row_idx, col_idx].set_title(mode)
            axes[row_idx, col_idx].axis("off")

    plt.suptitle("Leaf Gate Synthetic Negatives Preview (Proxy Only)", fontsize=14)
    plt.tight_layout()
    plt.savefig(SAVE_DIRS["figures"] / "leaf_gate_synthetic_negative_preview.png", dpi=200)
    plt.show()

leaf_gate_manifest_artifacts = build_leaf_gate_manifest(cfg, audit_artifacts["records"])
preview_synthetic_leaf_gate_examples(cfg, audit_artifacts["records"], n_per_mode=2)


## Section 7 — Leaf Gate Dataset And Loaders

In [ ]:
class LeafGateDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: CFG, is_train: bool):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.is_train = is_train

        self.positive_train_tfms = get_leaf_gate_positive_train_tfms(cfg.leaf_gate_img_size)
        self.eval_tfms = get_leaf_gate_eval_tfms(cfg.leaf_gate_img_size)
        self.negative_post_train_tfms = get_leaf_gate_negative_post_train_tfms()
        self.negative_post_eval_tfms = get_leaf_gate_negative_post_eval_tfms()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        image = load_rgb_image(row["path"])

        if int(row["leaf_label"]) == 1:
            tfms = self.positive_train_tfms if self.is_train else self.eval_tfms
            image = tfms(image=image)["image"]
        else:
            seed = None if self.is_train else int(row["deterministic_seed"])
            image = make_synthetic_negative_image(
                image=image,
                img_size=self.cfg.leaf_gate_img_size,
                mode=row["negative_mode"],
                seed=seed,
            )
            post_tfms = self.negative_post_train_tfms if self.is_train else self.negative_post_eval_tfms
            image = post_tfms(image=image)["image"]

        target = torch.tensor(float(row["leaf_label"]), dtype=torch.float32)
        return image, target

def build_leaf_gate_loaders(cfg: CFG, manifest_artifacts: Dict[str, pd.DataFrame]):
    train_df = manifest_artifacts["train"]
    val_df = manifest_artifacts["val"]
    test_df = manifest_artifacts["test"]

    train_ds = LeafGateDataset(train_df, cfg, is_train=True)
    val_ds = LeafGateDataset(val_df, cfg, is_train=False)
    test_ds = LeafGateDataset(test_df, cfg, is_train=False)

    generator = torch.Generator()
    generator.manual_seed(cfg.seed)

    common_loader_kwargs = dict(
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers and cfg.num_workers > 0,
        worker_init_fn=seed_worker,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.leaf_gate_batch_size,
        shuffle=True,
        generator=generator,
        **common_loader_kwargs,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.leaf_gate_batch_size,
        shuffle=False,
        **common_loader_kwargs,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.leaf_gate_batch_size,
        shuffle=False,
        **common_loader_kwargs,
    )

    print("Leaf gate loaders ready.")
    print("Train samples:", len(train_ds))
    print("Val samples  :", len(val_ds))
    print("Test samples :", len(test_ds))
    print("Train batches:", len(train_loader))
    print("Val batches  :", len(val_loader))
    print("Test batches :", len(test_loader))

    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

(
    leaf_gate_train_ds,
    leaf_gate_val_ds,
    leaf_gate_test_ds,
    leaf_gate_train_loader,
    leaf_gate_val_loader,
    leaf_gate_test_loader,
) = build_leaf_gate_loaders(cfg, leaf_gate_manifest_artifacts)


## Section 8 — Leaf Gate Model And Training

In [ ]:
def build_leaf_gate_model(cfg: CFG) -> nn.Module:
    model = timm.create_model(
        cfg.leaf_gate_model_name,
        pretrained=False,
        num_classes=1,
        drop_path_rate=0.05,
    )
    load_local_pretrained_weights(
        model=model,
        checkpoint_path=cfg.leaf_gate_pretrained_checkpoint,
        verbose_name="Leaf gate backbone",
    )
    return model

def binary_classification_metrics(y_true, y_prob, threshold: float = 0.5) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    acc = (tp + tn) / max(len(y_true), 1)
    false_accept_rate = fp / max(fp + tn, 1)
    false_reject_rate = fn / max(fn + tp, 1)

    return {
        "threshold": float(threshold),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "false_accept_rate": float(false_accept_rate),
        "false_reject_rate": float(false_reject_rate),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def train_leaf_gate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    scaler: GradScaler,
    grad_clip: float,
    grad_accum_steps: int = 1,
) -> Dict[str, float]:
    model.train()
    optimizer.zero_grad(set_to_none=True)

    all_targets = []
    all_probs = []
    loss_sum = 0.0
    total = 0

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast_ctx():
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)
            loss_for_backward = loss / grad_accum_steps

        scaler.scale(loss_for_backward).backward()

        if step % grad_accum_steps == 0 or step == len(loader):
            if grad_clip is not None and grad_clip > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

        batch_size = targets.size(0)
        loss_sum += loss.item() * batch_size
        total += batch_size

    metrics = binary_classification_metrics(all_targets, all_probs, threshold=0.5)
    metrics["loss"] = float(loss_sum / max(total, 1))
    return metrics

@torch.no_grad()
def evaluate_leaf_gate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = 0.5,
) -> Dict[str, object]:
    model.eval()

    all_targets = []
    all_probs = []
    loss_sum = 0.0
    total = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast_ctx():
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

        batch_size = targets.size(0)
        loss_sum += loss.item() * batch_size
        total += batch_size

    metrics = binary_classification_metrics(all_targets, all_probs, threshold=threshold)
    metrics["loss"] = float(loss_sum / max(total, 1))
    metrics["targets"] = np.asarray(all_targets).astype(int)
    metrics["probs"] = np.asarray(all_probs).astype(float)
    return metrics

def choose_leaf_gate_threshold(y_true, y_prob) -> Tuple[pd.DataFrame, float]:
    rows = []

    for threshold in np.round(np.arange(0.40, 0.96, 0.02), 2):
        metrics = binary_classification_metrics(y_true, y_prob, threshold=threshold)
        selection_score = metrics["f1"] - 0.50 * metrics["false_accept_rate"]
        rows.append({
            **metrics,
            "selection_score": float(selection_score),
        })

    threshold_df = pd.DataFrame(rows).sort_values(
        ["selection_score", "f1", "false_accept_rate"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    best_threshold = float(threshold_df.iloc[0]["threshold"])
    return threshold_df, best_threshold

def train_leaf_gate(
    cfg: CFG,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
) -> Dict[str, object]:
    out_dir = SAVE_DIRS["leaf_gate"]
    device = torch.device(cfg.device)

    model = build_leaf_gate_model(cfg).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=cfg.leaf_gate_lr,
        weight_decay=cfg.weight_decay,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.leaf_gate_epochs,
        eta_min=cfg.min_lr,
    )

    history = []
    best_score = -np.inf
    best_epoch = -1
    patience_counter = 0

    best_model_path = out_dir / "best_leaf_gate.pt"
    last_model_path = out_dir / "last_leaf_gate.pt"

    for epoch in range(1, cfg.leaf_gate_epochs + 1):
        start_time = time.time()

        train_metrics = train_leaf_gate_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=SCALER,
            grad_clip=cfg.grad_clip,
            grad_accum_steps=cfg.grad_accum_steps,
        )

        val_metrics = evaluate_leaf_gate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            threshold=0.5,
        )

        scheduler.step()
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]["lr"]

        selection_score = val_metrics["f1"] - 0.50 * val_metrics["false_accept_rate"]

        row = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_f1": val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_specificity": val_metrics["specificity"],
            "val_false_accept_rate": val_metrics["false_accept_rate"],
            "selection_score": selection_score,
            "time_sec": epoch_time,
        }
        history.append(row)

        print(
            f"LeafGate Epoch {epoch:03d}/{cfg.leaf_gate_epochs} | "
            f"LR {current_lr:.2e} | "
            f"TrainLoss {train_metrics['loss']:.4f} | "
            f"TrainF1 {train_metrics['f1']:.4f} | "
            f"ValLoss {val_metrics['loss']:.4f} | "
            f"ValF1 {val_metrics['f1']:.4f} | "
            f"ValFAR {val_metrics['false_accept_rate']:.4f} | "
            f"Time {epoch_time:.1f}s"
        )

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "cfg": asdict(cfg),
        }, last_model_path)

        if selection_score > best_score:
            best_score = selection_score
            best_epoch = epoch
            patience_counter = 0

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "cfg": asdict(cfg),
            }, best_model_path)

            print(f"Best leaf gate updated at epoch {epoch} | selection score = {best_score:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement | patience {patience_counter}/{cfg.leaf_gate_patience}")

        if patience_counter >= cfg.leaf_gate_patience:
            print(f"Leaf gate early stopping triggered. Best epoch: {best_epoch}")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(out_dir / "leaf_gate_history.csv", index=False)

    best_ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])

    val_metrics = evaluate_leaf_gate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        threshold=0.5,
    )

    threshold_df, best_threshold = choose_leaf_gate_threshold(
        y_true=val_metrics["targets"],
        y_prob=val_metrics["probs"],
    )
    threshold_df.to_csv(out_dir / "leaf_gate_threshold_sweep.csv", index=False)

    val_metrics_best_threshold = evaluate_leaf_gate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        threshold=best_threshold,
    )

    test_metrics_best_threshold = evaluate_leaf_gate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        threshold=best_threshold,
    )

    np.save(
        out_dir / "leaf_gate_val_confusion_matrix.npy",
        confusion_matrix(
            val_metrics_best_threshold["targets"],
            (val_metrics_best_threshold["probs"] >= best_threshold).astype(int),
            labels=[0, 1],
        ),
    )
    np.save(
        out_dir / "leaf_gate_test_confusion_matrix.npy",
        confusion_matrix(
            test_metrics_best_threshold["targets"],
            (test_metrics_best_threshold["probs"] >= best_threshold).astype(int),
            labels=[0, 1],
        ),
    )

    summary = {
        "note": "Leaf gate still uses synthetic negatives only in this version. Treat as a proxy gate, not true greenhouse rejection proof.",
        "best_epoch": int(best_epoch),
        "best_threshold": float(best_threshold),
        "val_f1": float(val_metrics_best_threshold["f1"]),
        "val_precision": float(val_metrics_best_threshold["precision"]),
        "val_recall": float(val_metrics_best_threshold["recall"]),
        "val_false_accept_rate": float(val_metrics_best_threshold["false_accept_rate"]),
        "synthetic_test_f1": float(test_metrics_best_threshold["f1"]),
        "synthetic_test_precision": float(test_metrics_best_threshold["precision"]),
        "synthetic_test_recall": float(test_metrics_best_threshold["recall"]),
        "synthetic_test_false_accept_rate": float(test_metrics_best_threshold["false_accept_rate"]),
        "best_model_path": str(best_model_path),
    }

    with open(out_dir / "leaf_gate_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nLeaf gate training finished.")
    print(json.dumps(summary, indent=2))

    return {
        "model": model,
        "summary": summary,
        "history_df": history_df,
        "best_model_path": str(best_model_path),
        "best_threshold": best_threshold,
        "threshold_df": threshold_df,
    }

leaf_gate_artifacts = train_leaf_gate(
    cfg=cfg,
    train_loader=leaf_gate_train_loader,
    val_loader=leaf_gate_val_loader,
    test_loader=leaf_gate_test_loader,
)

cfg.min_leaf_confidence = float(leaf_gate_artifacts["best_threshold"])
print("Updated cfg.min_leaf_confidence:", cfg.min_leaf_confidence)


## Section 9 — Disease Dataset And Loaders V3

In [ ]:
class DiseaseImageFolder(ImageFolder):
    def __init__(self, root: str, transform=None):
        super().__init__(root=root, transform=None)
        self.albu_transform = transform

    def __getitem__(self, index: int):
        path, label = self.samples[index]
        image = load_rgb_image(path)

        if self.albu_transform is not None:
            image = self.albu_transform(image=image)["image"]

        return image, label, path

def build_disease_datasets(cfg: CFG):
    root = Path(cfg.data_root)

    train_ds = DiseaseImageFolder(
        root=root / cfg.train_dir,
        transform=get_disease_train_tfms(cfg.disease_img_size),
    )
    val_ds = DiseaseImageFolder(
        root=root / cfg.val_dir,
        transform=get_disease_eval_tfms(cfg.disease_img_size),
    )
    test_ds = DiseaseImageFolder(
        root=root / cfg.test_dir,
        transform=get_disease_eval_tfms(cfg.disease_img_size),
    )

    return train_ds, val_ds, test_ds

def build_class_weight_table(train_ds: Dataset, cfg: CFG) -> pd.DataFrame:
    train_targets = np.array(train_ds.targets)
    counts = Counter(train_targets.tolist())

    rows = []
    for class_idx, class_name in enumerate(train_ds.classes):
        class_count = counts[class_idx]
        inverse_frequency = 1.0 / max(class_count, 1)
        powered_inverse_frequency = inverse_frequency ** cfg.class_weight_power
        rows.append({
            "class_idx": class_idx,
            "class_name": class_name,
            "train_count": class_count,
            "inverse_frequency": inverse_frequency,
            "powered_inverse_frequency": powered_inverse_frequency,
        })

    class_weights_df = pd.DataFrame(rows).sort_values("train_count", ascending=True).reset_index(drop=True)
    class_weights_df["normalized_powered_inverse_frequency"] = (
        class_weights_df["powered_inverse_frequency"] /
        class_weights_df["powered_inverse_frequency"].mean()
    )
    return class_weights_df

def build_class_weight_tensor(class_weights_df: pd.DataFrame) -> torch.Tensor:
    ordered = class_weights_df.sort_values("class_idx")
    weight_tensor = torch.tensor(
        ordered["normalized_powered_inverse_frequency"].values,
        dtype=torch.float32,
    )
    return weight_tensor

def build_disease_loaders(cfg: CFG):
    train_ds, val_ds, test_ds = build_disease_datasets(cfg)

    if len(train_ds.classes) != cfg.num_classes:
        raise ValueError(f"Expected {cfg.num_classes} classes, got {len(train_ds.classes)}")

    class_weights_df = build_class_weight_table(train_ds, cfg)
    class_weights_df.to_csv(SAVE_DIRS["disease_classifier"] / "class_weights_v3.csv", index=False)

    class_weight_tensor = build_class_weight_tensor(class_weights_df)

    generator = torch.Generator()
    generator.manual_seed(cfg.seed)

    common_loader_kwargs = dict(
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        persistent_workers=cfg.persistent_workers and cfg.num_workers > 0,
        worker_init_fn=seed_worker,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.disease_batch_size,
        shuffle=True,
        generator=generator,
        drop_last=False,
        **common_loader_kwargs,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.disease_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_kwargs,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.disease_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_kwargs,
    )

    print("Disease datasets and loaders are ready.")
    print("Classes      :", train_ds.classes)
    print("Train images :", len(train_ds))
    print("Val images   :", len(val_ds))
    print("Test images  :", len(test_ds))
    print("Train batches:", len(train_loader))
    print("Val batches  :", len(val_loader))
    print("Test batches :", len(test_loader))
    print("Weighted sampler enabled:", cfg.use_weighted_sampler)

    print("\nWeakest classes by training count:")
    display(class_weights_df.head(10))

    return {
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "class_names": train_ds.classes,
        "class_weights_df": class_weights_df,
        "class_weight_tensor": class_weight_tensor,
    }

disease_data = build_disease_loaders(cfg)
class_names = disease_data["class_names"]


## Section 10 — MixUp, CutMix, And Loss Utilities

In [ ]:
class SoftTargetCrossEntropy(nn.Module):
    def forward(self, logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.log_softmax(logits, dim=1)
        return -(soft_targets * log_probs).sum(dim=1).mean()

def smooth_one_hot(
    targets: torch.Tensor,
    num_classes: int,
    smoothing: float = 0.0,
) -> torch.Tensor:
    off_value = smoothing / num_classes
    on_value = 1.0 - smoothing + off_value

    y = torch.full(
        (targets.size(0), num_classes),
        fill_value=off_value,
        device=targets.device,
        dtype=torch.float32,
    )
    y.scatter_(1, targets.unsqueeze(1), on_value)
    return y

def build_mixed_soft_targets(
    targets_a: torch.Tensor,
    targets_b: torch.Tensor,
    lam: float,
    num_classes: int,
    smoothing: float = 0.0,
) -> torch.Tensor:
    y_a = smooth_one_hot(targets_a, num_classes=num_classes, smoothing=smoothing)
    y_b = smooth_one_hot(targets_b, num_classes=num_classes, smoothing=smoothing)
    return lam * y_a + (1.0 - lam) * y_b

def rand_bbox(size: Tuple[int, int, int, int], lam: float) -> Tuple[int, int, int, int]:
    _, _, h, w = size
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(w * cut_ratio)
    cut_h = int(h * cut_ratio)

    cx = np.random.randint(0, w)
    cy = np.random.randint(0, h)

    x1 = np.clip(cx - cut_w // 2, 0, w)
    x2 = np.clip(cx + cut_w // 2, 0, w)
    y1 = np.clip(cy - cut_h // 2, 0, h)
    y2 = np.clip(cy + cut_h // 2, 0, h)

    return x1, y1, x2, y2

def apply_mixup(
    images: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(images.size(0), device=images.device)

    mixed_images = lam * images + (1.0 - lam) * images[index]
    targets_a = targets
    targets_b = targets[index]
    return mixed_images, targets_a, targets_b, lam

def apply_cutmix(
    images: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(images.size(0), device=images.device)

    targets_a = targets
    targets_b = targets[index]

    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    mixed_images = images.clone()
    mixed_images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]

    patch_area = (x2 - x1) * (y2 - y1)
    total_area = images.size(-1) * images.size(-2)
    lam = 1.0 - (patch_area / max(total_area, 1))

    return mixed_images, targets_a, targets_b, float(lam)

def maybe_apply_mix_augmentation(
    images: torch.Tensor,
    targets: torch.Tensor,
    cfg: CFG,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float, str]:
    if images.size(0) < 2:
        return images, targets, targets, 1.0, "none"

    if random.random() > cfg.mix_prob:
        return images, targets, targets, 1.0, "none"

    available_modes = []
    if cfg.use_mixup and cfg.mixup_alpha > 0:
        available_modes.append("mixup")
    if cfg.use_cutmix and cfg.cutmix_alpha > 0:
        available_modes.append("cutmix")

    if not available_modes:
        return images, targets, targets, 1.0, "none"

    mode = random.choice(available_modes)

    if mode == "mixup":
        return (*apply_mixup(images, targets, cfg.mixup_alpha), "mixup")
    if mode == "cutmix":
        return (*apply_cutmix(images, targets, cfg.cutmix_alpha), "cutmix")

    return images, targets, targets, 1.0, "none"

def resolve_phase_name(cfg: CFG, epoch: int) -> str:
    phase_a_epochs = cfg.disease_epochs - cfg.mixup_off_last_n_epochs
    return "phase_a" if epoch <= phase_a_epochs else "phase_b"

print("MixUp / CutMix utilities are ready.")


## Section 11 — Disease Model Builder With Local Cached Backbone

In [ ]:
def build_disease_model(cfg: CFG) -> nn.Module:
    model = timm.create_model(
        cfg.disease_model_name,
        pretrained=False,
        num_classes=cfg.num_classes,
        drop_path_rate=cfg.drop_path_rate,
    )
    load_local_pretrained_weights(
        model=model,
        checkpoint_path=cfg.disease_pretrained_checkpoint,
        verbose_name="Disease backbone",
    )
    return model

def build_disease_criteria(
    cfg: CFG,
    class_weight_tensor: torch.Tensor,
    device: torch.device,
):
    hard_train_criterion = nn.CrossEntropyLoss(
        label_smoothing=cfg.label_smoothing,
    )

    finetune_criterion = nn.CrossEntropyLoss(
        weight=class_weight_tensor.to(device),
        label_smoothing=cfg.label_smoothing,
    )

    eval_criterion = nn.CrossEntropyLoss()
    soft_target_criterion = SoftTargetCrossEntropy()

    return {
        "hard_train": hard_train_criterion,
        "finetune": finetune_criterion,
        "eval": eval_criterion,
        "soft_target": soft_target_criterion,
    }

model_preview = build_disease_model(cfg).to(cfg.device)
xb, yb, pb = next(iter(disease_data["train_loader"]))
xb = xb.to(cfg.device, non_blocking=True)

with torch.no_grad(), autocast_ctx():
    out = model_preview(xb)

print("Preview input shape :", xb.shape)
print("Preview logits shape:", out.shape)

del model_preview
torch.cuda.empty_cache()


## Section 12 — EMA, Scheduler, Checkpointing, And Evaluation Helpers

In [ ]:
class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.ema_model = copy.deepcopy(model).eval()

        for param in self.ema_model.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        model_state = model.state_dict()
        ema_state = self.ema_model.state_dict()

        for key, ema_value in ema_state.items():
            model_value = model_state[key].detach()

            if not torch.is_floating_point(ema_value):
                ema_value.copy_(model_value)
            else:
                ema_value.mul_(self.decay).add_(model_value, alpha=1.0 - self.decay)

def build_disease_optimizer(cfg: CFG, model: nn.Module) -> optim.Optimizer:
    return optim.AdamW(
        model.parameters(),
        lr=cfg.disease_lr,
        weight_decay=cfg.weight_decay,
    )

def build_disease_epoch_lr_schedule(cfg: CFG) -> pd.DataFrame:
    phase_a_epochs = cfg.disease_epochs - cfg.mixup_off_last_n_epochs
    warmup_epochs = max(1, min(cfg.warmup_epochs, phase_a_epochs))

    rows = []
    for epoch in range(1, cfg.disease_epochs + 1):
        phase_name = resolve_phase_name(cfg, epoch)

        if phase_name == "phase_a":
            if epoch <= warmup_epochs:
                lr = cfg.disease_lr * epoch / warmup_epochs
            else:
                progress = (epoch - warmup_epochs) / max(1, phase_a_epochs - warmup_epochs)
                cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
                lr = cfg.finetune_lr + (cfg.disease_lr - cfg.finetune_lr) * cosine
        else:
            phase_b_idx = epoch - phase_a_epochs
            phase_b_total = cfg.mixup_off_last_n_epochs
            progress = (phase_b_idx - 1) / max(1, phase_b_total - 1)
            cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
            lr = cfg.min_lr + (cfg.finetune_lr - cfg.min_lr) * cosine

        rows.append({
            "epoch": epoch,
            "phase": phase_name,
            "lr": float(lr),
        })

    schedule_df = pd.DataFrame(rows)
    return schedule_df

def set_optimizer_lr(optimizer: optim.Optimizer, lr: float) -> None:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

@torch.no_grad()
def evaluate_disease_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    criterion: Optional[nn.Module],
    class_names: List[str],
) -> Dict[str, object]:
    model.eval()

    all_targets = []
    all_preds = []
    all_probs = []
    all_logits = []
    all_paths = []

    loss_sum = 0.0
    total = 0

    for images, targets, paths in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast_ctx():
            logits = model(images)
            loss = criterion(logits, targets) if criterion is not None else None

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        if loss is not None:
            batch_size = targets.size(0)
            loss_sum += loss.item() * batch_size
            total += batch_size
        else:
            total += targets.size(0)

        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.append(probs.cpu())
        all_logits.append(logits.cpu())
        all_paths.extend(list(paths))

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    probs_tensor = torch.cat(all_probs, dim=0) if len(all_probs) > 0 else torch.empty((0, len(class_names)))
    logits_tensor = torch.cat(all_logits, dim=0) if len(all_logits) > 0 else torch.empty((0, len(class_names)))

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    return {
        "loss": float(loss_sum / max(total, 1)) if criterion is not None else np.nan,
        "acc": float(acc),
        "balanced_acc": float(balanced_acc),
        "macro_f1": float(macro_f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs_tensor.numpy(),
        "logits_tensor": logits_tensor,
        "paths": all_paths,
        "class_names": class_names,
    }

def save_disease_checkpoint(
    path: Path,
    epoch: int,
    model: nn.Module,
    ema_helper: Optional[ModelEMA],
    cfg: CFG,
    class_names: List[str],
    optimizer: Optional[optim.Optimizer] = None,
    training_summary: Optional[Dict[str, object]] = None,
):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "cfg": asdict(cfg),
        "class_names": class_names,
    }

    if ema_helper is not None:
        payload["ema_state_dict"] = ema_helper.ema_model.state_dict()

    if optimizer is not None:
        payload["optimizer_state_dict"] = optimizer.state_dict()

    if training_summary is not None:
        payload["training_summary"] = training_summary

    torch.save(payload, path)

lr_schedule_df = build_disease_epoch_lr_schedule(cfg)
lr_schedule_df.to_csv(SAVE_DIRS["disease_classifier"] / "epoch_lr_schedule.csv", index=False)

print("Epoch LR schedule preview:")
display(lr_schedule_df.head(12))
display(lr_schedule_df.tail(8))


## Section 13 — Two-Phase Training Engine

In [ ]:
def train_disease_one_epoch(
    model: nn.Module,
    ema_helper: Optional[ModelEMA],
    loader: DataLoader,
    optimizer: optim.Optimizer,
    hard_train_criterion: nn.Module,
    finetune_criterion: nn.Module,      
    scaler: GradScaler,
    grad_clip: float,
    grad_accum_steps: int,
    cfg: CFG,
    epoch: int,
) -> Dict[str, float]:
    model.train()
    optimizer.zero_grad(set_to_none=True)

    phase_name = resolve_phase_name(cfg, epoch)

    loss_sum = 0.0
    total = 0
    correct = 0
    mix_batches = 0

    for step, batch in enumerate(loader, start=1):
        images, targets, _paths = batch
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        original_targets = targets

        with autocast_ctx():
            if phase_name == "phase_a":
                mixed_images, targets_a, targets_b, lam, mix_mode = maybe_apply_mix_augmentation(
                    images=images,
                    targets=targets,
                    cfg=cfg,
                )
                logits = model(mixed_images)

                if mix_mode == "none":
                    loss = hard_train_criterion(logits, original_targets)
                else:
                    soft_targets = build_mixed_soft_targets(
                        targets_a=targets_a,
                        targets_b=targets_b,
                        lam=lam,
                        num_classes=cfg.num_classes,
                        smoothing=cfg.label_smoothing,
                    )
                    loss = soft_target_criterion(logits, soft_targets)
                    mix_batches += 1
            else:
                logits = model(images)
                loss = finetune_criterion(logits, original_targets)

            loss_for_backward = loss / grad_accum_steps

        scaler.scale(loss_for_backward).backward()

        if step % grad_accum_steps == 0 or step == len(loader):
            if grad_clip is not None and grad_clip > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            if ema_helper is not None:
                ema_helper.update(model)

        preds = logits.argmax(dim=1)

        batch_size = original_targets.size(0)
        loss_sum += loss.item() * batch_size
        total += batch_size
        correct += (preds == original_targets).sum().item()

    return {
        "phase": phase_name,
        "loss": float(loss_sum / max(total, 1)),
        "acc_proxy": float(correct / max(total, 1)),
        "mix_batches": int(mix_batches),
    }

def run_disease_training(
    cfg: CFG,
    disease_data: Dict[str, object],
) -> Dict[str, object]:
    out_dir = SAVE_DIRS["disease_classifier"]
    device = torch.device(cfg.device)

    train_loader = disease_data["train_loader"]
    val_loader = disease_data["val_loader"]
    class_names = disease_data["class_names"]
    class_weight_tensor = disease_data["class_weight_tensor"]

    model = build_disease_model(cfg).to(device)
    criteria = build_disease_criteria(cfg, class_weight_tensor, device)
    optimizer = build_disease_optimizer(cfg, model)
    ema_helper = ModelEMA(model, decay=cfg.ema_decay) if cfg.use_ema else None

    history = []
    best_score = -np.inf
    best_epoch = -1
    patience_counter = 0

    best_model_path = out_dir / "best_disease_model.pt"
    last_model_path = out_dir / "last_disease_model.pt"

    for epoch in range(1, cfg.disease_epochs + 1):
        start_time = time.time()

        current_lr = float(lr_schedule_df.loc[lr_schedule_df["epoch"] == epoch, "lr"].iloc[0])
        set_optimizer_lr(optimizer, current_lr)

        train_metrics = train_disease_one_epoch(
            model=model,
            ema_helper=ema_helper,
            loader=train_loader,
            optimizer=optimizer,
            hard_train_criterion=criteria["hard_train"],
            finetune_criterion=criteria["finetune"],
            soft_target_criterion=criteria["soft_target"],
            device=device,
            scaler=SCALER,
            grad_clip=cfg.grad_clip,
            grad_accum_steps=cfg.grad_accum_steps,
            cfg=cfg,
            epoch=epoch,
        )

        raw_val_metrics = evaluate_disease_model(
            model=model,
            loader=val_loader,
            device=device,
            criterion=criteria["eval"],
            class_names=class_names,
        )

        if ema_helper is not None:
            ema_val_metrics = evaluate_disease_model(
                model=ema_helper.ema_model,
                loader=val_loader,
                device=device,
                criterion=criteria["eval"],
                class_names=class_names,
            )
        else:
            ema_val_metrics = raw_val_metrics

        selection_model_name = "ema" if cfg.use_ema else "raw"
        selection_val_metrics = ema_val_metrics if selection_model_name == "ema" else raw_val_metrics
        selection_score = selection_val_metrics["macro_f1"]

        epoch_time = time.time() - start_time

        row = {
            "epoch": epoch,
            "phase": train_metrics["phase"],
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_acc_proxy": train_metrics["acc_proxy"],
            "train_mix_batches": train_metrics["mix_batches"],
            "raw_val_loss": raw_val_metrics["loss"],
            "raw_val_acc": raw_val_metrics["acc"],
            "raw_val_macro_f1": raw_val_metrics["macro_f1"],
            "ema_val_loss": ema_val_metrics["loss"],
            "ema_val_acc": ema_val_metrics["acc"],
            "ema_val_macro_f1": ema_val_metrics["macro_f1"],
            "selection_model": selection_model_name,
            "selection_val_macro_f1": selection_score,
            "time_sec": epoch_time,
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d}/{cfg.disease_epochs} | "
            f"{train_metrics['phase']} | "
            f"LR {current_lr:.2e} | "
            f"TrainLoss {train_metrics['loss']:.4f} | "
            f"TrainAccProxy {train_metrics['acc_proxy']:.4f} | "
            f"RawValF1 {raw_val_metrics['macro_f1']:.4f} | "
            f"EMAValF1 {ema_val_metrics['macro_f1']:.4f} | "
            f"Time {epoch_time:.1f}s"
        )

        training_summary = {
            "epoch": epoch,
            "selection_model": selection_model_name,
            "selection_val_macro_f1": float(selection_score),
        }

        save_disease_checkpoint(
            path=last_model_path,
            epoch=epoch,
            model=model,
            ema_helper=ema_helper,
            cfg=cfg,
            class_names=class_names,
            optimizer=optimizer,
            training_summary=training_summary,
        )

        if selection_score > best_score:
            best_score = selection_score
            best_epoch = epoch
            patience_counter = 0

            save_disease_checkpoint(
                path=best_model_path,
                epoch=epoch,
                model=model,
                ema_helper=ema_helper,
                cfg=cfg,
                class_names=class_names,
                optimizer=optimizer,
                training_summary=training_summary,
            )

            print(
                f"Best disease checkpoint updated at epoch {epoch} | "
                f"{selection_model_name.upper()} Val Macro-F1 = {best_score:.4f}"
            )
        else:
            patience_counter += 1
            print(f"No improvement | patience {patience_counter}/{cfg.disease_patience}")

        if patience_counter >= cfg.disease_patience:
            print(f"Early stopping triggered. Best epoch: {best_epoch}")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(out_dir / "disease_history_v3.csv", index=False)

    best_ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])

    if ema_helper is not None and "ema_state_dict" in best_ckpt:
        ema_helper.ema_model.load_state_dict(best_ckpt["ema_state_dict"])

    training_summary = {
        "best_epoch": int(best_epoch),
        "best_selection_val_macro_f1": float(best_score),
        "selection_model": "ema" if cfg.use_ema else "raw",
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
    }

    with open(out_dir / "disease_training_summary_v3.json", "w", encoding="utf-8") as f:
        json.dump(training_summary, f, indent=2)

    print("\nDisease training finished.")
    print(json.dumps(training_summary, indent=2))

    return {
        "model": model,
        "ema_model": ema_helper.ema_model if ema_helper is not None else None,
        "history_df": history_df,
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "training_summary": training_summary,
        "selection_model": "ema" if cfg.use_ema else "raw",
        "eval_criterion": criteria["eval"],
    }

disease_training_artifacts = run_disease_training(cfg, disease_data)


## Section 14 — Raw Vs EMA Validation/Test Evaluation

In [ ]:
def compact_metric_summary(metrics: Dict[str, object]) -> Dict[str, float]:
    return {
        "loss": float(metrics["loss"]),
        "acc": float(metrics["acc"]),
        "balanced_acc": float(metrics["balanced_acc"]),
        "macro_f1": float(metrics["macro_f1"]),
    }

device = torch.device(cfg.device)
eval_criterion = disease_training_artifacts["eval_criterion"]

raw_val_metrics = evaluate_disease_model(
    model=disease_training_artifacts["model"],
    loader=disease_data["val_loader"],
    device=device,
    criterion=eval_criterion,
    class_names=class_names,
)

raw_test_metrics = evaluate_disease_model(
    model=disease_training_artifacts["model"],
    loader=disease_data["test_loader"],
    device=device,
    criterion=eval_criterion,
    class_names=class_names,
)

if disease_training_artifacts["ema_model"] is not None:
    ema_val_metrics = evaluate_disease_model(
        model=disease_training_artifacts["ema_model"],
        loader=disease_data["val_loader"],
        device=device,
        criterion=eval_criterion,
        class_names=class_names,
    )
    ema_test_metrics = evaluate_disease_model(
        model=disease_training_artifacts["ema_model"],
        loader=disease_data["test_loader"],
        device=device,
        criterion=eval_criterion,
        class_names=class_names,
    )
else:
    ema_val_metrics = raw_val_metrics
    ema_test_metrics = raw_test_metrics

selected_model_name = "ema" if ema_val_metrics["macro_f1"] >= raw_val_metrics["macro_f1"] else "raw"
selected_model = (
    disease_training_artifacts["ema_model"]
    if selected_model_name == "ema"
    else disease_training_artifacts["model"]
)
selected_val_metrics = ema_val_metrics if selected_model_name == "ema" else raw_val_metrics
selected_test_metrics = ema_test_metrics if selected_model_name == "ema" else raw_test_metrics

base_vs_ema_summary = {
    "selected_model_for_inference": selected_model_name,
    "raw_val": compact_metric_summary(raw_val_metrics),
    "raw_test": compact_metric_summary(raw_test_metrics),
    "ema_val": compact_metric_summary(ema_val_metrics),
    "ema_test": compact_metric_summary(ema_test_metrics),
}

with open(SAVE_DIRS["disease_classifier"] / "base_vs_ema_summary.json", "w", encoding="utf-8") as f:
    json.dump(base_vs_ema_summary, f, indent=2)

print("Base vs EMA evaluation:")
print(json.dumps(base_vs_ema_summary, indent=2))

disease_eval_artifacts = {
    "raw_val_metrics": raw_val_metrics,
    "raw_test_metrics": raw_test_metrics,
    "ema_val_metrics": ema_val_metrics,
    "ema_test_metrics": ema_test_metrics,
    "selected_model_name": selected_model_name,
    "selected_model": selected_model,
    "selected_val_metrics": selected_val_metrics,
    "selected_test_metrics": selected_test_metrics,
}


## Section 15 — Full-Dataset TTA Evaluation

In [ ]:
def get_eval_tta_transforms(img_size: int, tta_rounds: int) -> List[Tuple[str, A.Compose]]:
    transforms = [
        (
            "base",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "hflip",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.HorizontalFlip(p=1.0),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "zoom_in",
            A.Compose([
                A.Resize(int(img_size * 1.08), int(img_size * 1.08), interpolation=cv2.INTER_LINEAR),
                A.CenterCrop(img_size, img_size),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "bright",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.RandomBrightnessContrast(
                    brightness_limit=(0.06, 0.06),
                    contrast_limit=(0.03, 0.03),
                    p=1.0,
                ),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "dark",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.RandomBrightnessContrast(
                    brightness_limit=(-0.06, -0.06),
                    contrast_limit=(0.02, 0.02),
                    p=1.0,
                ),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "rot_plus",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.Affine(rotate=(7, 7), scale=(1.0, 1.0), shear=(0, 0), p=1.0),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
        (
            "rot_minus",
            A.Compose([
                A.Resize(img_size, img_size, interpolation=cv2.INTER_LINEAR),
                A.Affine(rotate=(-7, -7), scale=(1.0, 1.0), shear=(0, 0), p=1.0),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ]),
        ),
    ]

    if not cfg.use_tta:
        return transforms[:1]

    return transforms[:tta_rounds]

@torch.no_grad()
def evaluate_disease_model_tta(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    class_names: List[str],
    img_size: int,
    tta_rounds: int,
) -> Dict[str, object]:
    model.eval()
    tta_transforms = get_eval_tta_transforms(img_size=img_size, tta_rounds=tta_rounds)

    all_targets = []
    all_preds = []
    all_probs = []
    all_paths = []

    for _images, targets, paths in loader:
        batch_rgb = [load_rgb_image(path) for path in paths]
        tta_prob_tensors = []

        for _tta_name, tfms in tta_transforms:
            batch_tensor = torch.stack(
                [tfms(image=img)["image"] for img in batch_rgb],
                dim=0,
            ).to(device, non_blocking=True)

            with autocast_ctx():
                logits = model(batch_tensor)

            probs = torch.softmax(logits, dim=1)
            tta_prob_tensors.append(probs.detach().cpu())

        mean_probs = torch.stack(tta_prob_tensors, dim=0).mean(dim=0)
        preds = mean_probs.argmax(dim=1)

        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.append(mean_probs)
        all_paths.extend(list(paths))

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    probs_tensor = torch.cat(all_probs, dim=0) if len(all_probs) > 0 else torch.empty((0, len(class_names)))

    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    return {
        "acc": float(acc),
        "balanced_acc": float(balanced_acc),
        "macro_f1": float(macro_f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs_tensor.numpy(),
        "paths": all_paths,
        "class_names": class_names,
        "tta_rounds": int(len(tta_transforms)),
    }

selected_disease_model = disease_eval_artifacts["selected_model"]

print("Running full validation TTA evaluation...")
val_tta_metrics = evaluate_disease_model_tta(
    model=selected_disease_model,
    loader=disease_data["val_loader"],
    device=device,
    class_names=class_names,
    img_size=cfg.disease_img_size,
    tta_rounds=cfg.tta_rounds,
)

print("Running full test TTA evaluation...")
test_tta_metrics = evaluate_disease_model_tta(
    model=selected_disease_model,
    loader=disease_data["test_loader"],
    device=device,
    class_names=class_names,
    img_size=cfg.disease_img_size,
    tta_rounds=cfg.tta_rounds,
)

selected_base_val = disease_eval_artifacts["selected_val_metrics"]
selected_base_test = disease_eval_artifacts["selected_test_metrics"]

tta_summary = {
    "selected_model_for_tta": disease_eval_artifacts["selected_model_name"],
    "tta_rounds_used": int(val_tta_metrics["tta_rounds"]),
    "selected_base_val_acc": float(selected_base_val["acc"]),
    "selected_base_val_macro_f1": float(selected_base_val["macro_f1"]),
    "tta_val_acc": float(val_tta_metrics["acc"]),
    "tta_val_macro_f1": float(val_tta_metrics["macro_f1"]),
    "selected_base_test_acc": float(selected_base_test["acc"]),
    "selected_base_test_macro_f1": float(selected_base_test["macro_f1"]),
    "tta_test_acc": float(test_tta_metrics["acc"]),
    "tta_test_macro_f1": float(test_tta_metrics["macro_f1"]),
    "tta_val_acc_gain": float(val_tta_metrics["acc"] - selected_base_val["acc"]),
    "tta_val_macro_f1_gain": float(val_tta_metrics["macro_f1"] - selected_base_val["macro_f1"]),
    "tta_test_acc_gain": float(test_tta_metrics["acc"] - selected_base_test["acc"]),
    "tta_test_macro_f1_gain": float(test_tta_metrics["macro_f1"] - selected_base_test["macro_f1"]),
}

with open(SAVE_DIRS["disease_classifier"] / "full_dataset_tta_summary.json", "w", encoding="utf-8") as f:
    json.dump(tta_summary, f, indent=2)

print("Full-dataset TTA summary:")
print(json.dumps(tta_summary, indent=2))

tta_eval_artifacts = {
    "val_tta_metrics": val_tta_metrics,
    "test_tta_metrics": test_tta_metrics,
    "summary": tta_summary,
}


## Section 16 — Per-Class Error Analysis

In [ ]:
EXPERIMENT_2_WEAK_CLASSES = [
    "Pottassium_Deficiency",
    "Spotted_Wilt_Virus",
    "powdery_mildew",
    "Nitrogen_Deficiency",
    "Target_Spot",
]

def save_classification_outputs(
    out_dir: Path,
    metrics: Dict[str, object],
    split_name: str,
) -> Dict[str, object]:
    out_dir.mkdir(parents=True, exist_ok=True)

    y_true = metrics["y_true"]
    y_pred = metrics["y_pred"]
    probs = metrics["probs"]
    paths = metrics["paths"]
    class_names = metrics["class_names"]

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        output_dict=True,
        zero_division=0,
    )

    report_text = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(out_dir / f"{split_name}_classification_report.csv")

    with open(out_dir / f"{split_name}_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    np.save(out_dir / f"{split_name}_confusion_matrix.npy", cm)

    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{split_name.upper()} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(SAVE_DIRS["figures"] / f"{split_name}_confusion_matrix.png", dpi=200)
    plt.show()

    error_rows = []
    for idx, (true_idx, pred_idx, path_str) in enumerate(zip(y_true, y_pred, paths)):
        top_probs = probs[idx]
        top3_idx = np.argsort(top_probs)[::-1][:3]

        error_rows.append({
            "path": path_str,
            "true_idx": int(true_idx),
            "pred_idx": int(pred_idx),
            "true_class": class_names[int(true_idx)],
            "pred_class": class_names[int(pred_idx)],
            "is_correct": int(true_idx == pred_idx),
            "top1_prob": float(top_probs[int(pred_idx)]),
            "top1_class": class_names[int(pred_idx)],
            "top2_class": class_names[int(top3_idx[1])] if len(top3_idx) > 1 else "",
            "top2_prob": float(top_probs[int(top3_idx[1])]) if len(top3_idx) > 1 else np.nan,
            "top3_class": class_names[int(top3_idx[2])] if len(top3_idx) > 2 else "",
            "top3_prob": float(top_probs[int(top3_idx[2])]) if len(top3_idx) > 2 else np.nan,
        })

    errors_df = pd.DataFrame(error_rows)
    errors_df.to_csv(out_dir / f"{split_name}_predictions_detailed.csv", index=False)

    misclassified_df = errors_df[errors_df["is_correct"] == 0].copy()
    misclassified_df.to_csv(out_dir / f"{split_name}_misclassifications.csv", index=False)

    per_class_support_df = (
        errors_df.groupby("true_class")
        .agg(
            samples=("true_class", "size"),
            correct=("is_correct", "sum"),
        )
        .reset_index()
    )
    per_class_support_df["per_class_accuracy"] = (
        per_class_support_df["correct"] / per_class_support_df["samples"]
    )
    per_class_support_df = per_class_support_df.sort_values("per_class_accuracy", ascending=True)
    per_class_support_df.to_csv(out_dir / f"{split_name}_per_class_accuracy.csv", index=False)

    hardest_pairs_df = (
        misclassified_df.groupby(["true_class", "pred_class"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    hardest_pairs_df.to_csv(out_dir / f"{split_name}_hardest_confusions.csv", index=False)

    print(f"\n{split_name.upper()} report summary:")
    print(report_text)

    print(f"\nWorst {split_name} classes by per-class accuracy:")
    display(per_class_support_df.head(8))

    print(f"\nMost common {split_name} confusions:")
    display(hardest_pairs_df.head(12))

    return {
        "cm": cm,
        "report_df": report_df,
        "errors_df": errors_df,
        "misclassified_df": misclassified_df,
        "per_class_support_df": per_class_support_df,
        "hardest_pairs_df": hardest_pairs_df,
    }

def compare_per_class_with_experiment2(
    current_per_class_accuracy_df: pd.DataFrame,
    prev_experiment_root: str,
) -> Optional[pd.DataFrame]:
    prev_path = Path(prev_experiment_root) / "reports" / "test_per_class_accuracy.csv"
    if not prev_path.exists():
        print("Experiment 2 per-class accuracy file not found:", prev_path)
        return None

    prev_df = pd.read_csv(prev_path).rename(
        columns={"per_class_accuracy": "experiment2_per_class_accuracy"}
    )
    curr_df = current_per_class_accuracy_df.rename(
        columns={"per_class_accuracy": "experiment3_per_class_accuracy"}
    )

    merged_df = prev_df.merge(
        curr_df[["true_class", "experiment3_per_class_accuracy"]],
        on="true_class",
        how="outer",
    )

    merged_df["delta_accuracy"] = (
        merged_df["experiment3_per_class_accuracy"] - merged_df["experiment2_per_class_accuracy"]
    )
    merged_df = merged_df.sort_values("delta_accuracy").reset_index(drop=True)

    merged_df.to_csv(
        SAVE_DIRS["comparisons"] / "test_per_class_accuracy_vs_experiment2.csv",
        index=False,
    )

    weak_class_compare_df = merged_df[
        merged_df["true_class"].isin(EXPERIMENT_2_WEAK_CLASSES)
    ].sort_values("experiment3_per_class_accuracy")

    print("\nExperiment 3 vs Experiment 2 weak-class comparison:")
    display(weak_class_compare_df)

    return merged_df

selected_val_report_artifacts = save_classification_outputs(
    out_dir=SAVE_DIRS["reports"],
    metrics=disease_eval_artifacts["selected_val_metrics"],
    split_name=f"selected_{disease_eval_artifacts['selected_model_name']}_val",
)

selected_test_report_artifacts = save_classification_outputs(
    out_dir=SAVE_DIRS["reports"],
    metrics=disease_eval_artifacts["selected_test_metrics"],
    split_name=f"selected_{disease_eval_artifacts['selected_model_name']}_test",
)

per_class_compare_artifacts = compare_per_class_with_experiment2(
    current_per_class_accuracy_df=selected_test_report_artifacts["per_class_support_df"],
    prev_experiment_root=cfg.prev_experiment_root,
)


## Section 17 — Temperature Scaling On Selected Validation Logits

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self, init_temperature: float = 1.0):
        super().__init__()
        self.log_temperature = nn.Parameter(
            torch.log(torch.tensor([init_temperature], dtype=torch.float32))
        )

    @property
    def temperature(self) -> torch.Tensor:
        return self.log_temperature.exp().clamp(min=1e-3, max=10.0)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature

@torch.no_grad()
def collect_logits_targets_paths(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    model.eval()

    all_logits = []
    all_targets = []
    all_paths = []

    for images, targets, paths in loader:
        images = images.to(device, non_blocking=True)

        with autocast_ctx():
            logits = model(images)

        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())
        all_paths.extend(list(paths))

    logits_tensor = torch.cat(all_logits, dim=0)
    targets_tensor = torch.cat(all_targets, dim=0)
    return logits_tensor, targets_tensor, all_paths

def fit_temperature_scaler(
    logits: torch.Tensor,
    targets: torch.Tensor,
    out_dir: Path,
    init_temperature: float = 1.0,
) -> Dict[str, object]:
    device = torch.device(cfg.device)

    logits = logits.to(device)
    targets = targets.to(device)

    scaler_model = TemperatureScaler(init_temperature=init_temperature).to(device)
    criterion = nn.CrossEntropyLoss()

    before_nll = float(criterion(logits, targets).item())

    optimizer = optim.LBFGS(
        [scaler_model.log_temperature],
        lr=0.1,
        max_iter=100,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(scaler_model(logits), targets)
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        scaled_logits = scaler_model(logits)
        after_nll = float(criterion(scaled_logits, targets).item())
        learned_temperature = float(scaler_model.temperature.item())

    torch.save(
        {
            "log_temperature": scaler_model.log_temperature.detach().cpu(),
            "temperature": learned_temperature,
        },
        out_dir / "temperature_scaler.pt",
    )

    summary = {
        "before_nll": before_nll,
        "after_nll": after_nll,
        "temperature": learned_temperature,
    }

    with open(out_dir / "temperature_scaler_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("Temperature scaling finished.")
    print(json.dumps(summary, indent=2))

    return {
        "scaler_model": scaler_model,
        "summary": summary,
    }

selected_val_logits, selected_val_targets, selected_val_paths = collect_logits_targets_paths(
    model=disease_eval_artifacts["selected_model"],
    loader=disease_data["val_loader"],
    device=torch.device(cfg.device),
)

selected_test_logits, selected_test_targets, selected_test_paths = collect_logits_targets_paths(
    model=disease_eval_artifacts["selected_model"],
    loader=disease_data["test_loader"],
    device=torch.device(cfg.device),
)

calibration_artifacts = fit_temperature_scaler(
    logits=selected_val_logits,
    targets=selected_val_targets,
    out_dir=SAVE_DIRS["calibration"],
)

temperature_scaler = calibration_artifacts["scaler_model"]


## Section 18 — Reject / Uncertain Logic With Accepted-Only Test Metrics

In [ ]:
def softmax_numpy(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

@torch.no_grad()
def evaluate_disease_model_tta_calibrated(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    class_names: List[str],
    img_size: int,
    tta_rounds: int,
    temperature_scaler: Optional[TemperatureScaler] = None,
) -> Dict[str, object]:
    model.eval()
    if temperature_scaler is not None:
        temperature_scaler.eval()

    tta_transforms = get_eval_tta_transforms(img_size=img_size, tta_rounds=tta_rounds)

    all_targets = []
    all_preds = []
    all_probs = []
    all_paths = []

    for _images, targets, paths in loader:
        batch_rgb = [load_rgb_image(path) for path in paths]
        tta_prob_tensors = []

        for _tta_name, tfms in tta_transforms:
            batch_tensor = torch.stack(
                [tfms(image=img)["image"] for img in batch_rgb],
                dim=0,
            ).to(device, non_blocking=True)

            with autocast_ctx():
                logits = model(batch_tensor)

            if temperature_scaler is not None:
                logits = temperature_scaler(logits)

            probs = torch.softmax(logits, dim=1)
            tta_prob_tensors.append(probs.detach().cpu())

        mean_probs = torch.stack(tta_prob_tensors, dim=0).mean(dim=0)
        preds = mean_probs.argmax(dim=1)

        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.append(mean_probs)
        all_paths.extend(list(paths))

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    probs_tensor = torch.cat(all_probs, dim=0) if len(all_probs) > 0 else torch.empty((0, len(class_names)))

    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    return {
        "acc": float(acc),
        "balanced_acc": float(balanced_acc),
        "macro_f1": float(macro_f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs_tensor.numpy(),
        "paths": all_paths,
        "class_names": class_names,
        "tta_rounds": int(len(tta_transforms)),
    }

def evaluate_reject_policy(
    probs: np.ndarray,
    y_true: np.ndarray,
    min_confidence: float,
    min_margin: float,
) -> Dict[str, float]:
    top1_idx = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]

    top1_prob = sorted_probs[:, 0]
    top2_prob = sorted_probs[:, 1] if probs.shape[1] > 1 else np.zeros_like(top1_prob)
    margin = top1_prob - top2_prob

    accepted_mask = (top1_prob >= min_confidence) & (margin >= min_margin)
    coverage = float(accepted_mask.mean())

    if accepted_mask.sum() == 0:
        accepted_macro_f1 = 0.0
        accepted_acc = 0.0
    else:
        accepted_macro_f1 = float(
            f1_score(y_true[accepted_mask], top1_idx[accepted_mask], average="macro")
        )
        accepted_acc = float(
            accuracy_score(y_true[accepted_mask], top1_idx[accepted_mask])
        )

    selection_score = accepted_macro_f1 * (coverage ** 0.35)

    return {
        "min_confidence": float(min_confidence),
        "min_margin": float(min_margin),
        "coverage": coverage,
        "accepted_macro_f1": accepted_macro_f1,
        "accepted_acc": accepted_acc,
        "selection_score": float(selection_score),
    }

def choose_disease_reject_thresholds(
    probs: np.ndarray,
    y_true: np.ndarray,
) -> Dict[str, object]:
    rows = []

    for min_confidence in np.round(np.arange(0.55, 0.96, 0.02), 2):
        for min_margin in np.round(np.arange(0.01, 0.21, 0.01), 2):
            metrics = evaluate_reject_policy(
                probs=probs,
                y_true=y_true,
                min_confidence=float(min_confidence),
                min_margin=float(min_margin),
            )
            rows.append(metrics)

    threshold_df = pd.DataFrame(rows)

    viable_df = threshold_df[threshold_df["coverage"] >= 0.80].copy()
    if viable_df.empty:
        viable_df = threshold_df.copy()

    viable_df = viable_df.sort_values(
        ["selection_score", "accepted_macro_f1", "coverage"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    best_row = viable_df.iloc[0].to_dict()

    threshold_df.to_csv(SAVE_DIRS["thresholds"] / "disease_threshold_sweep_v3.csv", index=False)

    with open(SAVE_DIRS["thresholds"] / "disease_threshold_summary_v3.json", "w", encoding="utf-8") as f:
        json.dump(best_row, f, indent=2)

    return {
        "threshold_df": threshold_df,
        "best_row": best_row,
    }

def summarize_acceptance_metrics(
    probs: np.ndarray,
    y_true: np.ndarray,
    min_confidence: float,
    min_margin: float,
) -> Dict[str, object]:
    top1_idx = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]

    top1_prob = sorted_probs[:, 0]
    top2_prob = sorted_probs[:, 1] if probs.shape[1] > 1 else np.zeros_like(top1_prob)
    margin = top1_prob - top2_prob

    accepted_mask = (top1_prob >= min_confidence) & (margin >= min_margin)
    rejected_mask = ~accepted_mask

    if accepted_mask.sum() == 0:
        accepted_acc = 0.0
        accepted_macro_f1 = 0.0
    else:
        accepted_acc = float(accuracy_score(y_true[accepted_mask], top1_idx[accepted_mask]))
        accepted_macro_f1 = float(f1_score(y_true[accepted_mask], top1_idx[accepted_mask], average="macro"))

    return {
        "coverage": float(accepted_mask.mean()),
        "reject_rate": float(rejected_mask.mean()),
        "accepted_count": int(accepted_mask.sum()),
        "rejected_count": int(rejected_mask.sum()),
        "accepted_acc": accepted_acc,
        "accepted_macro_f1": accepted_macro_f1,
    }

print("Running calibrated validation TTA evaluation...")
calibrated_val_tta_metrics = evaluate_disease_model_tta_calibrated(
    model=disease_eval_artifacts["selected_model"],
    loader=disease_data["val_loader"],
    device=device,
    class_names=class_names,
    img_size=cfg.disease_img_size,
    tta_rounds=cfg.tta_rounds,
    temperature_scaler=temperature_scaler,
)

print("Running calibrated test TTA evaluation...")
calibrated_test_tta_metrics = evaluate_disease_model_tta_calibrated(
    model=disease_eval_artifacts["selected_model"],
    loader=disease_data["test_loader"],
    device=device,
    class_names=class_names,
    img_size=cfg.disease_img_size,
    tta_rounds=cfg.tta_rounds,
    temperature_scaler=temperature_scaler,
)

calibrated_val_tta_report_artifacts = save_classification_outputs(
    out_dir=SAVE_DIRS["reports"],
    metrics=calibrated_val_tta_metrics,
    split_name="selected_calibrated_tta_val",
)

calibrated_test_tta_report_artifacts = save_classification_outputs(
    out_dir=SAVE_DIRS["reports"],
    metrics=calibrated_test_tta_metrics,
    split_name="selected_calibrated_tta_test",
)

disease_threshold_artifacts = choose_disease_reject_thresholds(
    probs=calibrated_val_tta_metrics["probs"],
    y_true=calibrated_val_tta_metrics["y_true"],
)

cfg.min_disease_confidence = float(disease_threshold_artifacts["best_row"]["min_confidence"])
cfg.uncertainty_margin = float(disease_threshold_artifacts["best_row"]["min_margin"])

val_acceptance_metrics = summarize_acceptance_metrics(
    probs=calibrated_val_tta_metrics["probs"],
    y_true=calibrated_val_tta_metrics["y_true"],
    min_confidence=cfg.min_disease_confidence,
    min_margin=cfg.uncertainty_margin,
)

test_acceptance_metrics = summarize_acceptance_metrics(
    probs=calibrated_test_tta_metrics["probs"],
    y_true=calibrated_test_tta_metrics["y_true"],
    min_confidence=cfg.min_disease_confidence,
    min_margin=cfg.uncertainty_margin,
)

reject_eval_summary = {
    "min_leaf_confidence": float(cfg.min_leaf_confidence),
    "min_disease_confidence": float(cfg.min_disease_confidence),
    "uncertainty_margin": float(cfg.uncertainty_margin),
    "validation_accepted_only": val_acceptance_metrics,
    "test_accepted_only": test_acceptance_metrics,
    "note": "Accepted-only metrics below are computed on the tomato-leaf test set. Leaf-gate rejection is separate and not evaluated here because this test split contains leaf images only.",
}

with open(SAVE_DIRS["thresholds"] / "accepted_only_test_metrics_v3.json", "w", encoding="utf-8") as f:
    json.dump(reject_eval_summary, f, indent=2)

print("Updated calibrated reject thresholds:")
print("  min_disease_confidence:", cfg.min_disease_confidence)
print("  uncertainty_margin    :", cfg.uncertainty_margin)

print("\nAccepted-only metrics summary:")
print(json.dumps(reject_eval_summary, indent=2))

reject_eval_artifacts = {
    "calibrated_val_tta_metrics": calibrated_val_tta_metrics,
    "calibrated_test_tta_metrics": calibrated_test_tta_metrics,
    "threshold_artifacts": disease_threshold_artifacts,
    "val_acceptance_metrics": val_acceptance_metrics,
    "test_acceptance_metrics": test_acceptance_metrics,
    "summary": reject_eval_summary,
}


## Section 19 — Final Real-Image Inference Using Selected Model + TTA

In [ ]:
def format_topk_predictions(
    probs: np.ndarray,
    class_names: List[str],
    top_k: int = 3,
) -> List[Dict[str, object]]:
    top_indices = np.argsort(probs)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "class_idx": int(idx),
            "class_name": class_names[int(idx)],
            "probability": float(probs[int(idx)]),
            "percentage": float(probs[int(idx)] * 100.0),
        })
    return results

def apply_reject_logic(
    leaf_probability: float,
    disease_probs: np.ndarray,
    class_names: List[str],
    cfg: CFG,
) -> Dict[str, object]:
    topk = format_topk_predictions(
        probs=disease_probs,
        class_names=class_names,
        top_k=cfg.top_k,
    )

    top1 = topk[0]
    top2 = topk[1] if len(topk) > 1 else {"probability": 0.0, "class_name": ""}
    margin = float(top1["probability"] - top2["probability"])

    if leaf_probability < cfg.min_leaf_confidence:
        return {
            "status": "reject_no_leaf",
            "message": "Tomato leaf not found",
            "accepted": False,
            "leaf_probability": float(leaf_probability),
            "topk_predictions": topk,
            "predicted_class": None,
            "top1_probability": float(top1["probability"]),
            "margin": margin,
        }

    if float(top1["probability"]) < cfg.min_disease_confidence:
        return {
            "status": "reject_low_confidence",
            "message": "Leaf visible but disease unclear",
            "accepted": False,
            "leaf_probability": float(leaf_probability),
            "topk_predictions": topk,
            "predicted_class": None,
            "top1_probability": float(top1["probability"]),
            "margin": margin,
        }

    if margin < cfg.uncertainty_margin:
        return {
            "status": "reject_ambiguous",
            "message": "Low-confidence prediction",
            "accepted": False,
            "leaf_probability": float(leaf_probability),
            "topk_predictions": topk,
            "predicted_class": None,
            "top1_probability": float(top1["probability"]),
            "margin": margin,
        }

    return {
        "status": "accepted",
        "message": f"Predicted disease: {top1['class_name']}",
        "accepted": True,
        "leaf_probability": float(leaf_probability),
        "topk_predictions": topk,
        "predicted_class": top1["class_name"],
        "top1_probability": float(top1["probability"]),
        "margin": margin,
    }

@torch.no_grad()
def predict_leaf_probability(
    leaf_gate_model: nn.Module,
    image_rgb: np.ndarray,
    cfg: CFG,
) -> float:
    leaf_gate_model.eval()

    tfms = get_leaf_gate_eval_tfms(cfg.leaf_gate_img_size)
    tensor = tfms(image=image_rgb)["image"].unsqueeze(0).to(cfg.device, non_blocking=True)

    with autocast_ctx():
        logits = leaf_gate_model(tensor).squeeze(1)

    prob = torch.sigmoid(logits).item()
    return float(prob)

@torch.no_grad()
def predict_disease_probabilities_tta_calibrated(
    disease_model: nn.Module,
    image_rgb: np.ndarray,
    cfg: CFG,
    temperature_scaler: Optional[TemperatureScaler],
) -> Dict[str, object]:
    disease_model.eval()
    if temperature_scaler is not None:
        temperature_scaler.eval()

    tta_transforms = get_eval_tta_transforms(cfg.disease_img_size, cfg.tta_rounds)

    prob_list = []
    per_tta_rows = []

    for tta_name, tfms in tta_transforms:
        tensor = tfms(image=image_rgb)["image"].unsqueeze(0).to(cfg.device, non_blocking=True)

        with autocast_ctx():
            logits = disease_model(tensor)

        if temperature_scaler is not None:
            logits = temperature_scaler(logits)

        probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
        prob_list.append(probs)

        per_tta_rows.append({
            "tta_name": tta_name,
            "top1_class": class_names[int(np.argmax(probs))],
            "top1_probability": float(np.max(probs)),
        })

    mean_probs = np.mean(np.stack(prob_list, axis=0), axis=0)

    return {
        "mean_probs": mean_probs,
        "per_tta_df": pd.DataFrame(per_tta_rows),
    }

def run_full_inference(
    image_path: str,
    leaf_gate_model: nn.Module,
    disease_model: nn.Module,
    temperature_scaler: Optional[TemperatureScaler],
    cfg: CFG,
    class_names: List[str],
    save_visual: bool = True,
) -> Dict[str, object]:
    image_path = str(image_path)
    image_rgb = load_rgb_image(image_path)

    leaf_probability = predict_leaf_probability(
        leaf_gate_model=leaf_gate_model,
        image_rgb=image_rgb,
        cfg=cfg,
    )

    disease_pred = predict_disease_probabilities_tta_calibrated(
        disease_model=disease_model,
        image_rgb=image_rgb,
        cfg=cfg,
        temperature_scaler=temperature_scaler,
    )

    decision = apply_reject_logic(
        leaf_probability=leaf_probability,
        disease_probs=disease_pred["mean_probs"],
        class_names=class_names,
        cfg=cfg,
    )

    result = {
        "image_path": image_path,
        "status": decision["status"],
        "message": decision["message"],
        "accepted": bool(decision["accepted"]),
        "leaf_probability": float(decision["leaf_probability"]),
        "predicted_class": decision["predicted_class"],
        "top1_probability": float(decision["top1_probability"]),
        "margin": float(decision["margin"]),
        "topk_predictions": decision["topk_predictions"],
        "per_tta": disease_pred["per_tta_df"].to_dict(orient="records"),
    }

    if save_visual:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(image_rgb)
        ax.axis("off")

        title = (
            f"{result['message']}\n"
            f"Leaf={result['leaf_probability'] * 100:.1f}% | "
            f"Top1={result['top1_probability'] * 100:.1f}%"
        )

        ax.set_title(title)
        plt.tight_layout()

        figure_name = Path(image_path).stem + "_experiment3_deployment_preview.png"
        plt.savefig(SAVE_DIRS["predictions"] / figure_name, dpi=180)
        plt.show()

    return result

def batch_inference_on_test_subset(
    disease_data: Dict[str, object],
    leaf_gate_model: nn.Module,
    disease_model: nn.Module,
    temperature_scaler: Optional[TemperatureScaler],
    cfg: CFG,
    class_names: List[str],
    sample_size: int = 40,
) -> pd.DataFrame:
    test_paths = [sample[0] for sample in disease_data["test_ds"].samples]
    sampled_paths = random.sample(test_paths, k=min(sample_size, len(test_paths)))

    rows = []
    for image_path in sampled_paths:
        result = run_full_inference(
            image_path=image_path,
            leaf_gate_model=leaf_gate_model,
            disease_model=disease_model,
            temperature_scaler=temperature_scaler,
            cfg=cfg,
            class_names=class_names,
            save_visual=False,
        )

        row = {
            "image_path": result["image_path"],
            "status": result["status"],
            "message": result["message"],
            "accepted": result["accepted"],
            "leaf_probability": result["leaf_probability"],
            "predicted_class": result["predicted_class"],
            "top1_probability": result["top1_probability"],
            "margin": result["margin"],
        }

        for item in result["topk_predictions"]:
            row[f"top{item['rank']}_class"] = item["class_name"]
            row[f"top{item['rank']}_probability"] = item["probability"]

        rows.append(row)

    predictions_df = pd.DataFrame(rows)
    predictions_df.to_csv(
        SAVE_DIRS["predictions"] / "experiment3_test_subset_deployment_predictions.csv",
        index=False,
    )
    return predictions_df

deployment_preview_df = batch_inference_on_test_subset(
    disease_data=disease_data,
    leaf_gate_model=leaf_gate_artifacts["model"],
    disease_model=disease_eval_artifacts["selected_model"],
    temperature_scaler=temperature_scaler,
    cfg=cfg,
    class_names=class_names,
    sample_size=40,
)

display(deployment_preview_df.head(10))


## Section 20 — Final Experiment 3 Summary With Side-By-Side Comparison Vs Experiment 2

In [ ]:
def maybe_load_json(path: Path) -> Optional[Dict[str, object]]:
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_final_experiment3_summary(
    cfg: CFG,
    leaf_gate_artifacts: Dict[str, object],
    disease_training_artifacts: Dict[str, object],
    disease_eval_artifacts: Dict[str, object],
    tta_eval_artifacts: Dict[str, object],
    calibration_artifacts: Dict[str, object],
    reject_eval_artifacts: Dict[str, object],
    deployment_preview_df: pd.DataFrame,
) -> Dict[str, object]:
    prev_disease_summary_path = Path(cfg.prev_experiment_root) / "disease_classifier" / "disease_summary.json"
    prev_deployment_summary_path = Path(cfg.prev_experiment_root) / "deployment" / "final_deployment_summary.json"

    experiment2_disease_summary = maybe_load_json(prev_disease_summary_path)
    experiment2_deployment_summary = maybe_load_json(prev_deployment_summary_path)

    selected_base_test = disease_eval_artifacts["selected_test_metrics"]
    calibrated_tta_test = reject_eval_artifacts["calibrated_test_tta_metrics"]
    accepted_only_test = reject_eval_artifacts["test_acceptance_metrics"]

    summary = {
        "experiment_root": str(SAVE_DIRS["root"]),
        "dataset_root": cfg.data_root,
        "selected_model_for_inference": disease_eval_artifacts["selected_model_name"],
        "disease_model_name": cfg.disease_model_name,
        "leaf_gate_model_name": cfg.leaf_gate_model_name,
        "disease_img_size": cfg.disease_img_size,
        "leaf_gate_img_size": cfg.leaf_gate_img_size,
        "disease_lr": cfg.disease_lr,
        "finetune_lr": cfg.finetune_lr,
        "weight_decay": cfg.weight_decay,
        "drop_path_rate": cfg.drop_path_rate,
        "mixup_off_last_n_epochs": cfg.mixup_off_last_n_epochs,
        "use_mixup": cfg.use_mixup,
        "use_cutmix": cfg.use_cutmix,
        "use_ema": cfg.use_ema,
        "ema_decay": cfg.ema_decay,
        "tta_rounds": cfg.tta_rounds,
        "best_disease_model_path": disease_training_artifacts["best_model_path"],
        "best_leaf_gate_model_path": leaf_gate_artifacts["best_model_path"],
        "temperature": calibration_artifacts["summary"]["temperature"],
        "min_leaf_confidence": cfg.min_leaf_confidence,
        "min_disease_confidence": cfg.min_disease_confidence,
        "uncertainty_margin": cfg.uncertainty_margin,
        "selected_base_test_acc": float(selected_base_test["acc"]),
        "selected_base_test_macro_f1": float(selected_base_test["macro_f1"]),
        "calibrated_tta_test_acc": float(calibrated_tta_test["acc"]),
        "calibrated_tta_test_macro_f1": float(calibrated_tta_test["macro_f1"]),
        "accepted_only_test_acc": float(accepted_only_test["accepted_acc"]),
        "accepted_only_test_macro_f1": float(accepted_only_test["accepted_macro_f1"]),
        "accepted_only_test_coverage": float(accepted_only_test["coverage"]),
        "accepted_only_test_reject_rate": float(accepted_only_test["reject_rate"]),
        "leaf_gate_summary": leaf_gate_artifacts["summary"],
        "training_summary": disease_training_artifacts["training_summary"],
        "deployment_preview_accept_rate": float(deployment_preview_df["accepted"].mean()) if len(deployment_preview_df) > 0 else None,
        "deployment_preview_status_counts": deployment_preview_df["status"].value_counts(dropna=False).to_dict(),
        "notes": [
            "Experiment 3 is generalization-first: no weighted sampler, two-phase training, EMA, calibrated TTA.",
            "Leaf gate is still a synthetic proxy and not a true real-world non-leaf validation.",
            "Accepted-only metrics are computed on the tomato-leaf test set after calibrated disease rejection thresholds.",
        ],
    }

    if experiment2_disease_summary is not None:
        summary["experiment2_comparison"] = {
            "experiment2_test_acc": float(experiment2_disease_summary["final_test_acc"]),
            "experiment2_test_macro_f1": float(experiment2_disease_summary["final_test_macro_f1"]),
            "experiment3_base_test_acc": float(selected_base_test["acc"]),
            "experiment3_base_test_macro_f1": float(selected_base_test["macro_f1"]),
            "experiment3_calibrated_tta_test_acc": float(calibrated_tta_test["acc"]),
            "experiment3_calibrated_tta_test_macro_f1": float(calibrated_tta_test["macro_f1"]),
            "delta_base_test_acc_vs_exp2": float(selected_base_test["acc"] - experiment2_disease_summary["final_test_acc"]),
            "delta_base_test_macro_f1_vs_exp2": float(selected_base_test["macro_f1"] - experiment2_disease_summary["final_test_macro_f1"]),
            "delta_calibrated_tta_test_acc_vs_exp2": float(calibrated_tta_test["acc"] - experiment2_disease_summary["final_test_acc"]),
            "delta_calibrated_tta_test_macro_f1_vs_exp2": float(calibrated_tta_test["macro_f1"] - experiment2_disease_summary["final_test_macro_f1"]),
        }

    if experiment2_deployment_summary is not None:
        summary["experiment2_deployment_reference"] = {
            "experiment2_min_leaf_confidence": experiment2_deployment_summary.get("min_leaf_confidence"),
            "experiment2_min_disease_confidence": experiment2_deployment_summary.get("min_disease_confidence"),
            "experiment2_uncertainty_margin": experiment2_deployment_summary.get("uncertainty_margin"),
            "experiment2_preview_accept_rate": experiment2_deployment_summary.get("preview_accept_rate"),
        }

    with open(SAVE_DIRS["deployment"] / "final_experiment3_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    if "experiment2_comparison" in summary:
        pd.DataFrame([summary["experiment2_comparison"]]).to_csv(
            SAVE_DIRS["comparisons"] / "experiment2_vs_experiment3_summary.csv",
            index=False,
        )

    pd.DataFrame([{
        "selected_base_test_acc": summary["selected_base_test_acc"],
        "selected_base_test_macro_f1": summary["selected_base_test_macro_f1"],
        "calibrated_tta_test_acc": summary["calibrated_tta_test_acc"],
        "calibrated_tta_test_macro_f1": summary["calibrated_tta_test_macro_f1"],
        "accepted_only_test_acc": summary["accepted_only_test_acc"],
        "accepted_only_test_macro_f1": summary["accepted_only_test_macro_f1"],
        "accepted_only_test_coverage": summary["accepted_only_test_coverage"],
    }]).to_csv(
        SAVE_DIRS["deployment"] / "final_experiment3_metrics.csv",
        index=False,
    )

    with open(SAVE_DIRS["deployment"] / "class_names.json", "w", encoding="utf-8") as f:
        json.dump(class_names, f, indent=2)

    print("Final Experiment 3 summary saved to:", SAVE_DIRS["deployment"] / "final_experiment3_summary.json")
    return summary

final_experiment3_summary = build_final_experiment3_summary(
    cfg=cfg,
    leaf_gate_artifacts=leaf_gate_artifacts,
    disease_training_artifacts=disease_training_artifacts,
    disease_eval_artifacts=disease_eval_artifacts,
    tta_eval_artifacts=tta_eval_artifacts,
    calibration_artifacts=calibration_artifacts,
    reject_eval_artifacts=reject_eval_artifacts,
    deployment_preview_df=deployment_preview_df,
)

print(json.dumps(final_experiment3_summary, indent=2))


In [ ]:
from pathlib import Path
import json
import shutil
import torch

PROJECT_ROOT = Path(".")
EXPERIMENT3_ROOT = PROJECT_ROOT / "trackB_outputs" / "Track_B_Experiment_3"

FRIEND_BUNDLE_DIR = EXPERIMENT3_ROOT / "friend_bundle_simple"
MODELS_DIR = FRIEND_BUNDLE_DIR / "models"
META_DIR = FRIEND_BUNDLE_DIR / "meta"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

exp3_final_summary_path = EXPERIMENT3_ROOT / "deployment" / "final_experiment3_summary.json"
exp3_best_ckpt_path = EXPERIMENT3_ROOT / "disease_classifier" / "best_disease_model.pt"
leaf_gate_ckpt_path = EXPERIMENT3_ROOT / "leaf_gate" / "best_leaf_gate.pt"
temperature_scaler_path = EXPERIMENT3_ROOT / "calibration" / "temperature_scaler.pt"

with open(exp3_final_summary_path, "r", encoding="utf-8") as f:
    exp3_final = json.load(f)

best_ckpt = torch.load(exp3_best_ckpt_path, map_location="cpu", weights_only=False)

if "ema_state_dict" in best_ckpt:
    selected_state_dict = best_ckpt["ema_state_dict"]
    selected_method = "ema_light_tta_with_temperature"
else:
    selected_state_dict = best_ckpt["model_state_dict"]
    selected_method = "raw_light_tta_with_temperature"

class_names = best_ckpt.get("class_names")
if class_names is None:
    raise KeyError("class_names not found in Experiment 3 checkpoint.")

disease_model_export_path = MODELS_DIR / "disease_model.pt"
leaf_gate_export_path = MODELS_DIR / "leaf_gate_model.pt"
temperature_export_path = MODELS_DIR / "temperature_scaler.pt"

torch.save(
    {
        "model_state_dict": selected_state_dict,
        "class_names": class_names,
        "source": str(exp3_best_ckpt_path),
        "selected_method": selected_method,
    },
    disease_model_export_path,
)

shutil.copy2(leaf_gate_ckpt_path, leaf_gate_export_path)
shutil.copy2(temperature_scaler_path, temperature_export_path)

class_names_path = META_DIR / "class_names.json"
thresholds_path = META_DIR / "thresholds.json"
inference_config_path = META_DIR / "inference_config.json"
readme_path = FRIEND_BUNDLE_DIR / "README_for_flutter.txt"

with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

thresholds_payload = {
    "min_leaf_confidence": float(exp3_final.get("min_leaf_confidence", 0.40)),
    "min_disease_confidence": float(exp3_final.get("min_disease_confidence", 0.67)),
    "uncertainty_margin": float(exp3_final.get("uncertainty_margin", 0.01)),
    "top_k": 3,
}
with open(thresholds_path, "w", encoding="utf-8") as f:
    json.dump(thresholds_payload, f, indent=2)

inference_config = {
    "selected_export_method": selected_method,
    "disease_model": "models/disease_model.pt",
    "leaf_gate_model": "models/leaf_gate_model.pt",
    "temperature_scaler": "models/temperature_scaler.pt",
    "apply_temperature_before_softmax": True,
    "use_tta": True,
    "tta_rounds": 3,
    "class_names_file": "meta/class_names.json",
    "thresholds_file": "meta/thresholds.json",
}
with open(inference_config_path, "w", encoding="utf-8") as f:
    json.dump(inference_config, f, indent=2)

readme_text = f"""Experiment 3 Simple Friend Bundle

Load these:
1. models/leaf_gate_model.pt
2. models/disease_model.pt
3. models/temperature_scaler.pt
4. meta/class_names.json
5. meta/thresholds.json
6. meta/inference_config.json

Inference:
- Use the disease model from models/disease_model.pt
- Apply temperature scaling before softmax
- Use light TTA
- Output class order must match meta/class_names.json
"""

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("Experiment 3 simple friend bundle created.")
print(FRIEND_BUNDLE_DIR)
print()
print("Files:")
for path in sorted(FRIEND_BUNDLE_DIR.rglob("*")):
    if path.is_file():
        print(path)
